# 🔴 ARES — Mars Route Planner v5 (fast)
**Like a map app:** pick START and END on Mars, press **Find paths**.

**New in v5**
- **Fast:** one search per feature instead of dozens of generators, merge tests and pruning.
- **3 routes only:** every feature (shortest, least climb, flattest, smoothest, most clearance, least sand, lowest radiation, balanced) gets one route; the best overall is chosen and highlighted in green, plus the 2 most useful alternatives.
- **Layers explain the route:** each layer shows its heat map and the route built only for that layer (dotted cyan). New layer *Why: total cost* shows the mixed price the best route follows.

Steps: run cells 1-3, choose the area in cell 4, run cell 5. Honest limits: radiation is a proxy; zones are hypothetical; grid cells are finer than the 200 m source data.


In [ ]:
# Cell 1 · Setup (installs the one extra tool needed to read Mars height files)
import importlib, subprocess, sys
def _ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True); importlib.import_module(mod or pkg); return True
        except Exception: return False
HAVE_RASTERIO = _ensure("rasterio")
for _p, _m in [("numpy", None), ("scipy", None), ("matplotlib", None), ("pandas", None), ("pillow", "PIL")]: _ensure(_p, _m)
import math, json, time, heapq, io, os, warnings, contextlib, html as _html
import numpy as np, pandas as pd
from scipy import ndimage as ndi
from matplotlib.colors import LightSource
from IPython.display import display, HTML
print("Ready. rasterio (real height files):", "yes" if HAVE_RASTERIO else "NO - real data cannot load, demo terrain will be used")


In [ ]:
# ===== 02_settings =====
def S(key, label, default, unit, group, tag, note, kind="num", lo=None, hi=None, options=None, info=False):
    return dict(key=key, label=label, default=default, unit=unit, group=group, tag=tag, note=note,
                kind=kind, lo=lo, hi=hi, options=options, info=info)

# SET 1 · TERRAIN RULES — stage 1 only. No crew, oxygen or battery here.
TR_SPEC = [
    S("risky_slope_deg", "Slope where it gets risky", 15, "deg", "Slope and safety", "THRESHOLD", "Steeper ground is flagged and counts in 'steep %'.", lo=3, hi=40),
    S("max_slope_deg", "Slope that is impassable", 18, "deg", "Slope and safety", "THRESHOLD", "HARD rule: never walked.", lo=5, hi=40),
    S("danger_buffer_m", "Safety buffer around impassable ground", 10, "m", "Slope and safety", "THRESHOLD", "HARD rule: stay this far away.", lo=0, hi=500),
    S("rough_threshold_m", "Rough-ground threshold", 2.0, "m", "Slope and safety", "THRESHOLD", "Height wobble that counts as fully rough.", lo=0.1, hi=50),
    S("max_grid_cells", "Map size (cells, long side)", 160, "cells", "Search and picture", "APP SETTING", "Bigger = finer but slower.", lo=40, hi=400),
    S("weighted_eps", "Weighted A* strength", 2.5, "x", "Search and picture", "APP SETTING", "Higher = faster, less exact.", lo=1.0, hi=10),
    S("leg_length_m", "Leg length in leg table", 500, "m", "Search and picture", "APP SETTING", "Also split where ground type changes.", lo=100, hi=5000),
    S("sun_az_deg", "Sun direction (azimuth)", 315, "deg", "Search and picture", "APP SETTING", "For shaded relief and shadow layer.", lo=0, hi=360),
    S("sun_elev_deg", "Sun height", 25, "deg", "Search and picture", "APP SETTING", "For shaded relief and shadow layer.", lo=2, hi=80),
    S("shadow_level", "Shadow threshold", 0.15, "0-1", "Search and picture", "APP SETTING", "Darker than this = 'shadow'.", lo=0.0, hi=1.0),
    S("mars_gravity", "Mars gravity", 3.71, "m/s²", "Search and picture", "KNOWN CONSTANT", "Information only.", info=True),
    S("rad_shield_pct", "Max shielding by terrain", 50, "%", "Radiation layer (proxy)", "ASSUMPTION", "Radiation index of a fully enclosed spot vs open ground. 50 = half the open-ground exposure. A guess: not measured data.", lo=0, hi=90),
    S("rad_alt_pct_km", "Radiation change per km of height", 5, "%/km", "Radiation layer (proxy)", "ASSUMPTION", "Higher ground = thinner air above = a little more radiation. A guess: not measured data.", lo=0, hi=30),
]

# SET 2 · ASSUMPTIONS — stage 2 only. Guessed numbers about crew and suit.
A_SPEC = [
    S("crew_size", "Crew size", 2, "people", "Crew and suit", "ASSUMPTION", "Totals are multiplied by this.", lo=1, hi=6),
    S("suit_mass_kg", "Suit + person mass", 130, "kg", "Crew and suit", "ASSUMPTION", "Used by the energy formula.", lo=60, hi=250),
    S("flat_speed_kmh", "Walking speed on flat ground", 2.5, "km/h", "Crew and suit", "ASSUMPTION", "Other slopes follow Tobler's rule.", lo=0.5, hi=6),
    S("slowest_walker_factor", "Slowest walker speed factor", 1.0, "x", "Crew and suit", "ASSUMPTION", "0.8 = team 20% slower.", lo=0.3, hi=1.5),
    S("eva_limit_h", "EVA time limit", 8, "h", "Crew and suit", "ASSUMPTION", "Hard rule.", lo=1, hi=24),
    S("o2_tank_l", "Oxygen tank size", 1200, "L", "Oxygen", "ASSUMPTION", "Per person.", lo=100, hi=5000),
    S("start_o2_pct", "Starting oxygen", 100, "%", "Oxygen", "ASSUMPTION", "Tank fill at start.", lo=1, hi=100),
    S("o2_rest_l_min", "Oxygen use at rest", 0.8, "L/min", "Oxygen", "ASSUMPTION", "Flat base use.", lo=0.1, hi=3),
    S("o2_slope_coeff", "Extra oxygen per sin(slope)", 2.5, "L/min", "Oxygen", "ASSUMPTION", "use = rest + this × sin(slope).", lo=0, hi=10),
    S("o2_kj_per_l", "Energy per litre of oxygen", 20.9, "kJ/L", "Oxygen", "STANDARD RULE", "Approximate body-chemistry rule.", lo=15, hi=25),
    S("o2_model", "Oxygen formula", "conservative", "", "Oxygen", "APP SETTING", "slope / energy (Minetti) / conservative = larger of the two.", kind="choice", options=["conservative", "slope", "energy"]),
    S("o2_warn_pct", "Oxygen warning level", 80, "% used", "Oxygen", "ASSUMPTION", "Warn at this usage.", lo=10, hi=100),
    S("o2_reserve_pct", "Oxygen reserve to walk back", 20, "% of tank", "Oxygen", "ASSUMPTION", "Hard rule.", lo=0, hi=60),
    S("battery_kwh", "Battery size", 4.0, "kWh", "Battery and heating", "ASSUMPTION", "Per suit.", lo=0.5, hi=20),
    S("start_batt_pct", "Starting battery", 100, "%", "Battery and heating", "ASSUMPTION", "Fill at start.", lo=1, hi=100),
    S("life_support_w", "Life-support draw", 150, "W", "Battery and heating", "ASSUMPTION", "Always on.", lo=0, hi=2000),
    S("heater_w", "Heater power", 350, "W", "Battery and heating", "ASSUMPTION", "Continuous.", lo=0, hi=2000),
    S("other_w", "Other power use", 0, "W", "Battery and heating", "ASSUMPTION", "Radio, motors...", lo=0, hi=2000),
    S("cold_extra_w", "Extra heating in shadow", 0, "W", "Battery and heating", "ASSUMPTION", "Shadow is only a proxy. 0 = off.", lo=0, hi=1000),
    S("batt_reserve_pct", "Battery reserve to walk back", 20, "% of battery", "Battery and heating", "ASSUMPTION", "Hard rule.", lo=0, hi=60),
    S("rad_msv_day", "Radiation dose rate", 0.67, "mSv/day", "Radiation", "ASSUMPTION", "Dose rate on open ground (radiation index 1.0). Not in the 5 resources: a guess.", lo=0, hi=5),
    S("rad_warn_msv", "Radiation warning level", 0, "mSv", "Radiation", "ASSUMPTION", "0 = no warning.", lo=0, hi=50),
]


class Settings:
    """One group of settings. Reading S["key"] is recorded so we can audit what was really used."""
    def __init__(self, name, spec):
        self.name = name
        self.spec = {s["key"]: s for s in spec}
        self.values = {k: s["default"] for k, s in self.spec.items()}
        self.used = set()

    def __getitem__(self, k):
        self.used.add(k)
        return self.values[k]

    def set(self, k, v):
        d = self.spec[k]["default"]
        if self.spec[k]["kind"] == "choice": pass
        elif isinstance(d, bool): v = bool(v)
        elif isinstance(d, int): v = int(round(float(v)))
        elif isinstance(d, float): v = float(v)
        self.values[k] = v

    def reset(self):
        for k, s in self.spec.items():
            self.values[k] = s["default"]

    def keys_numeric(self):
        return [k for k, s in self.spec.items() if s["kind"] == "num" and not s["info"]]

    @contextlib.contextmanager
    def temp(self, **kw):
        old = {k: self.values[k] for k in kw}
        try:
            self.values.update(kw); yield
        finally:
            self.values.update(old)

    def table(self):
        rows = []
        for k, s in self.spec.items():
            status = "info only" if s["info"] else ("used" if k in self.used else "not used in this run")
            rows.append(dict(Setting=s["label"], Value=self.values[k], Unit=s["unit"], Tag=s["tag"], Group=s["group"], Status=status, key=k))
        return pd.DataFrame(rows)


TR = Settings("Terrain rules", TR_SPEC)
A = Settings("Assumptions", A_SPEC)

# Route styles = saved price weights (a search hyperparameter, not physics).
PRESETS = {
    "Balanced":      dict(slope=3.0, roughness=1.5, sand=2.5, near_danger=3.0, climb=2.0, radiation=1.5, height=1.0, slope_cap_deg=18),
    "Safe":          dict(slope=6.0, roughness=2.0, sand=4.0, near_danger=6.0, climb=1.0, radiation=3.0, slope_cap_deg=8),
    "Quick":         dict(slope=1.0, roughness=0.5, sand=1.0, near_danger=1.0, climb=0.2, radiation=0.3, slope_cap_deg=15),
    "Low-energy":    dict(slope=2.0, roughness=0.5, sand=2.5, near_danger=1.0, climb=6.0, radiation=0.3, slope_cap_deg=18),
    "Low-radiation": dict(slope=1.5, roughness=0.5, sand=1.5, near_danger=1.5, climb=1.0, radiation=10.0, slope_cap_deg=18),
}
print(f"{len([s for s in TR_SPEC if not s['info']])} terrain rules · {len(A_SPEC)} assumptions · {len(PRESETS)} route styles loaded.")

# ===== 03_data =====
RESOURCES = [
    dict(n=1, name="NASA JPL Mars Trek", url="https://trek.nasa.gov/mars/", role="Whole-planet photo map + DEM file you export"),
    dict(n=2, name="USGS-NASA Planetary Geologic Mapping", url="https://astrogeology-usgs.hub.arcgis.com/pages/interactive-maps-mars", role="Sand / rock-unit mask you export"),
    dict(n=3, name="Curiosity's proposed path up Mount Sharp", url="https://science.nasa.gov/resource/curiositys-proposed-path-up-mount-sharp/", role="Reference route (waypoints you type in)"),
    dict(n=4, name="NASA Worldview", url="https://worldview.earthdata.nasa.gov/", role="NOT USED (Earth data)"),
    dict(n=5, name="Perseverance Raw Images", url="https://mars.nasa.gov/mars2020/multimedia/raw-images/", role="NOT USED (Jezero photos)"),
]

R_MARS = 3396190.0                      # m, radius of the Mars sphere used by the USGS/Trek equirectangular maps
M_PER_DEG = R_MARS * math.pi / 180.0    # metres per degree of latitude

# Trek basemap tile address (WMTS). The layer name is from memory and must be CONFIRMED on trek.nasa.gov — if it fails the app draws an offline picture.
TREK_TILE_URL = "https://trek.nasa.gov/tiles/Mars/EQ/Mars_Viking_MDIM21_ClrMosaic_global_232m/1.0.0/default/default028mm/{z}/{y}/{x}.jpg"

DEFAULT_AREA = dict(lat_min=-4.78, lat_max=-4.60, lon_min=137.35, lon_max=137.55)   # Gale crater box from the plan (Q3 default)


def area_size_m(area):
    lat0 = 0.5 * (area["lat_min"] + area["lat_max"])
    h = (area["lat_max"] - area["lat_min"]) * M_PER_DEG
    w = (area["lon_max"] - area["lon_min"]) * M_PER_DEG * math.cos(math.radians(lat0))
    return w, h


class Terrain:
    """Height grid Z (metres) on a square grid of cell size px (metres), plus optional sand mask."""
    def __init__(self, Z, px, area, label, is_demo, sand=None, sand_label="no sand data", notes=None, native_res_m=None):
        self.Z, self.px, self.area, self.label, self.is_demo = Z, px, dict(area), label, is_demo
        self.sand, self.sand_label, self.notes, self.native_res_m = sand, sand_label, list(notes or []), native_res_m
        self.H, self.W = Z.shape

    def rc_to_lonlat(self, r, c):
        a = self.area
        lat = a["lat_max"] - (np.asarray(r, float) + 0.5) / self.H * (a["lat_max"] - a["lat_min"])
        lon = a["lon_min"] + (np.asarray(c, float) + 0.5) / self.W * (a["lon_max"] - a["lon_min"])
        return lon, lat

    def lonlat_to_rc(self, lon, lat):
        a = self.area
        r = (a["lat_max"] - lat) / (a["lat_max"] - a["lat_min"]) * self.H - 0.5
        c = (lon - a["lon_min"]) / (a["lon_max"] - a["lon_min"]) * self.W - 0.5
        return int(np.clip(round(r), 0, self.H - 1)), int(np.clip(round(c), 0, self.W - 1))


def _fill_nan(a):
    m = np.isnan(a)
    if not m.any():
        return a
    idx = ndi.distance_transform_edt(m, return_distances=False, return_indices=True)
    return a[tuple(idx)]


def make_demo_terrain(area, n):
    """DEMO ONLY: made-up hills + a bowl + sand patches, so the app can run with no download."""
    rng = np.random.default_rng(7)
    w_m, h_m = area_size_m(area)
    px = max(w_m, h_m) / n
    W_, H_ = max(30, int(round(w_m / px))), max(30, int(round(h_m / px)))
    yy, xx = np.mgrid[0:H_, 0:W_]
    u, v = xx / W_, yy / H_
    size = max(w_m, h_m)
    mound = np.exp(-(((u - 0.62) / 0.30) ** 2 + ((v - 0.40) / 0.26) ** 2))
    bowl = -np.exp(-(((u - 0.25) / 0.17) ** 2 + ((v - 0.72) / 0.17) ** 2))
    def noise(s):
        g = ndi.gaussian_filter(rng.normal(size=(H_, W_)), s)
        return g / g.std()
    Z = 2000.0 + size * 0.072 * mound + size * 0.02 * bowl + size * (0.006 * noise(18) + 0.0025 * noise(7) + 0.00035 * noise(2))
    gy, gx = np.gradient(Z, px)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    sn = noise(10)
    sand = ((sn > np.quantile(sn, 0.78)) & (slope < 9)).astype(float)
    return Terrain(Z, px, area, "DEMO terrain (synthetic, not Mars)", True, sand=sand, sand_label="SYNTHETIC sand patches (demo)",
                   notes=["DEMO: heights and sand are made up. Real results need a DEM exported from Mars Trek."])


_DS = {}
def _open_cached(path):
    import rasterio
    if path not in _DS: _DS[path] = rasterio.open(path)   # header is fetched once, not on every load
    return _DS[path]


def _read_dem_file(path, area, shape):
    if not HAVE_RASTERIO:
        raise RuntimeError("Loading a DEM file needs rasterio.  Run:  pip install rasterio")
    import rasterio
    from rasterio.windows import from_bounds
    from rasterio.enums import Resampling
    ds = _open_cached(path)
    if True:
        if ds.crs is None or ds.crs.is_geographic:
            bounds = (area["lon_min"], area["lat_min"], area["lon_max"], area["lat_max"])
        else:      # equirectangular metres (Trek / USGS Mars maps)
            bounds = (R_MARS * math.radians(area["lon_min"]), R_MARS * math.radians(area["lat_min"]),
                      R_MARS * math.radians(area["lon_max"]), R_MARS * math.radians(area["lat_max"]))
        win = from_bounds(*bounds, transform=ds.transform)
        arr = ds.read(1, window=win, out_shape=shape, resampling=Resampling.bilinear, boundless=True, fill_value=np.nan).astype(float)
        if ds.nodata is not None:
            arr[arr == ds.nodata] = np.nan
        native = abs(ds.transform.a) if not ds.crs.is_geographic else abs(ds.transform.a) * M_PER_DEG
    arr[np.abs(arr) > 1e6] = np.nan
    if np.isnan(arr).mean() > 0.5:
        raise RuntimeError("More than half of the chosen area is outside the DEM file (or empty). Check the lat/lon box and the file.")
    return _fill_nan(arr), native


def _read_mask_file(path, shape):
    from PIL import Image
    try:
        im = Image.open(path).convert("L").resize((shape[1], shape[0]), Image.NEAREST)
        a = np.asarray(im, float) / 255.0
    except Exception:
        if not HAVE_RASTERIO:
            raise
        import rasterio
        from rasterio.enums import Resampling
        with rasterio.open(path) as ds:
            a = ds.read(1, out_shape=shape, resampling=Resampling.nearest).astype(float)
        a = (a - np.nanmin(a)) / (np.nanmax(a) - np.nanmin(a) + 1e-9)
    return (a > 0.5).astype(float)


DATA_SOURCES = ["Demo terrain (synthetic)", "Mars Trek DEM file (GeoTIFF you exported)"]


def load_terrain(source, area, max_grid, dem_file="", sand_file=""):
    w_m, h_m = area_size_m(area)
    if w_m <= 0 or h_m <= 0:
        raise ValueError("Area is empty: lat_max must be larger than lat_min, lon_max larger than lon_min.")
    if source.startswith("Demo"):
        T = make_demo_terrain(area, max_grid)
        if sand_file:
            T.sand = _read_mask_file(sand_file, T.Z.shape); T.sand_label = f"sand mask file: {os.path.basename(sand_file)}"
        return T
    if not dem_file or not os.path.exists(dem_file):
        raise FileNotFoundError("DEM file not found. Export a height map (GeoTIFF) from Mars Trek, then type its path, or switch to Demo terrain.")
    px = max(w_m, h_m) / max_grid
    shape = (max(30, int(round(h_m / px))), max(30, int(round(w_m / px))))
    Z, native = _read_dem_file(dem_file, area, shape)
    sand, sl = None, "no sand data"
    if sand_file:
        sand = _read_mask_file(sand_file, shape); sl = f"sand mask file: {os.path.basename(sand_file)}"
    notes = []
    if native and native > px * 1.5:
        notes.append(f"DEM native resolution ≈ {native:.0f} m is coarser than the grid ({px:.0f} m): small steep spots are smoothed away, so slopes are UNDERESTIMATED.")
    return Terrain(Z, px, area, f"Mars Trek DEM: {os.path.basename(dem_file)}", False, sand=sand, sand_label=sl, notes=notes, native_res_m=native)


def fetch_global_basemap(zoom=2, timeout=8):
    """Whole-planet photo map from Mars Trek tiles. Returns an RGB array, or None if the tiles cannot be loaded."""
    try:
        import requests
        from PIL import Image
        cols, rows = 2 ** (zoom + 1), 2 ** zoom
        canvas = Image.new("RGB", (cols * 256, rows * 256))
        for y in range(rows):
            for x in range(cols):
                r = requests.get(TREK_TILE_URL.format(z=zoom, y=y, x=x), timeout=timeout)
                r.raise_for_status()
                canvas.paste(Image.open(io.BytesIO(r.content)).convert("RGB"), (x * 256, y * 256))
        return np.asarray(canvas)
    except Exception as e:
        print("Mars Trek tiles could not be loaded (", type(e).__name__, ") → using the offline picture. Check the tile layer name on trek.nasa.gov.")
        return None


def parse_reference(text):
    """'-4.7,137.4; -4.65,137.5' → [(lat,lon),...].  Waypoints you read off the Curiosity path page (resource 3)."""
    pts = []
    for chunk in text.replace("\n", ";").split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue
        a, b = [float(v) for v in chunk.replace(" ", ",").split(",") if v != ""][:2]
        pts.append((a, b))
    return pts


def data_check_table(T, ref_pts, basemap_ok):
    rows = []
    rows.append([1, RESOURCES[0]["name"], "photo map: " + ("loaded" if basemap_ok else "offline picture") +
                 " · height: " + ("DEMO (synthetic)" if T and T.is_demo else (T.label if T else "not loaded"))])
    rows.append([2, RESOURCES[1]["name"], T.sand_label if T else "not loaded"])
    rows.append([3, RESOURCES[2]["name"], f"{len(ref_pts)} reference waypoints typed in" if ref_pts else "no reference route given"])
    rows.append([4, RESOURCES[3]["name"], "not used (Earth data)"])
    rows.append([5, RESOURCES[4]["name"], "not used (Jezero photos)"])
    return pd.DataFrame(rows, columns=["#", "Resource", "Status"])


# ---- v3 extras: optional photo (to click on) + layer guide
Terrain.photo = None
Terrain.photo_label = "no photo loaded (the picture is drawn from the height map)"


def load_photo(T, path):
    """A real picture (CTX / HiRISE / Trek export) covering EXACTLY the lat/lon box. Background only."""
    from PIL import Image as PILImage
    im = PILImage.open(path).convert("RGB")
    if max(im.size) > 1400:
        im.thumbnail((1400, 1400))
    T.photo = np.asarray(im)
    T.photo_label = f"photo: {os.path.basename(path)} (assumed to cover exactly the lat/lon box)"
    return T


LAYER_GUIDE = pd.DataFrame([
    ["Photo (optional)", "Resource 1 · Trek photo export", "colour", "Real surface colour: bright orange = dust, dark = rock/sand. NOT height.", "Background to click on. Never used in a calculation."],
    ["Height (DEM)", "Resource 1 · Trek DEM GeoTIFF", "metres", "Colour ramp low → high.", "Base of everything."],
    ["Shaded relief", "height + sun angle", "0-1", "Bright = faces the sun.", "Picture and shadow layer."],
    ["Slope", "steepest step to 8 neighbours", "degrees", "Dark flat → bright steep.", "Hazard, price, risk, steep %."],
    ["Roughness", "local height spread, slope removed", "metres", "Dark smooth → bright bumpy.", "Price, rough %."],
    ["Sand mask (optional)", "Resource 2 · USGS map export", "0/1", "Tan = sand.", "Price, sand %."],
    ["Impassable", "slope > limit", "yes/no", "Red.", "Hard rule."],
    ["Clearance", "distance to impassable cell", "metres", "Yellow band near red.", "Buffer, near-danger price and %."],
    ["Blocked", "impassable OR inside buffer", "yes/no", "Dark red.", "Cells the search may not use."],
    ["Closeness", "from clearance", "0-1", "Yellow strength.", "Soft price, risk."],
    ["Shadow (proxy)", "shaded relief < threshold", "yes/no", "Blue.", "Extra heating in stage 2 only (if > 0 W)."],
    ["Radiation (proxy)", "sky view from the height map + height + zones you draw", "index, 1.0 = open ground", "Blue = shielded, red = hot zone.", "Route price, radiation exposure, stage 2 dose."],
    ["Route agreement", "how many routes pass through each square", "count", "Brighter = more routes share this corridor.", "Showing shared corridors and decision points. Not used in any calculation."],
], columns=["Layer", "Where it comes from", "Unit", "What the colours mean", "Used for"])

# ===== 04_layers =====
NB8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


class Layers:
    pass


def local_std(a, size):
    m = ndi.uniform_filter(a, size)
    return np.sqrt(np.maximum(ndi.uniform_filter(a * a, size) - m * m, 0))


def build_layers(T, A):
    L = Layers()
    Z, px = T.Z, T.px
    H, Wd = Z.shape
    Zp = np.pad(Z, 1, mode="edge")
    s = np.zeros_like(Z)
    for dr, dc in NB8:
        nb = Zp[1 + dr:1 + dr + H, 1 + dc:1 + dc + Wd]
        s = np.maximum(s, np.abs(Z - nb) / (px * math.hypot(dr, dc)))
    L.slope = np.degrees(np.arctan(s))                       # degrees, steepest step out of each square
    L.rough = local_std(Z - ndi.gaussian_filter(Z, 2.0), 5)  # metres of height wobble (slope removed)
    L.has_sand = T.sand is not None
    L.sand = T.sand if L.has_sand else np.zeros_like(Z)
    L.hill = LightSource(azdeg=A["sun_az_deg"], altdeg=A["sun_elev_deg"]).hillshade(Z, vert_exag=1, dx=px, dy=px)
    L.shadow = L.hill < A["shadow_level"]                    # PROXY for shadow
    L.danger = L.slope > A["max_slope_deg"]                  # impassable ground
    if L.danger.any():
        L.clearance = ndi.distance_transform_edt(~L.danger) * px
    else:
        L.clearance = np.full(Z.shape, 1e4)
    buf = A["danger_buffer_m"]
    L.blocked = L.danger | (L.clearance < buf)               # HARD rule (never a weight)
    L.influence = max(5 * buf, 2 * px, 1.0)
    L.closeness = np.clip(1 - L.clearance / L.influence, 0, 1)
    return L

ALGOS = ['Dijkstra', 'A*', 'Weighted A*', 'Greedy', 'A* + smoothing']

import base64 as _b64


def _enc(a, dtype):
    return _b64.b64encode(np.ascontiguousarray(a, dtype=dtype).tobytes()).decode()


def export_payload(T, L):
    """Everything the browser app needs: the height grid and the terrain layers (numbers), plus the settings and route styles."""
    spec = lambda sp: [dict(key=s["key"], label=s["label"], default=s["default"], unit=s["unit"], group=s["group"], tag=s["tag"], note=s["note"],
                            kind=s["kind"], lo=s["lo"], hi=s["hi"], options=s["options"], info=s["info"]) for s in sp]
    a = T.area
    return dict(H=int(T.H), W=int(T.W), px=float(T.px), area={k: float(v) for k, v in a.items()},
                Z=_enc(T.Z, "<f8"), slope=_enc(L.slope, "<f4"), rough=_enc(L.rough, "<f4"), hill=_enc(L.hill, "<f4"), sand=_enc(L.sand, "u1"),
                hasSand=bool(L.has_sand), isDemo=bool(T.is_demo), label=T.label, sandLabel=T.sand_label, notes=list(T.notes),
                photo=None, photoLabel=T.photo_label, TR_SPEC=spec(TR_SPEC), A_SPEC=spec(A_SPEC), PRESETS=PRESETS, ALGOS=list(ALGOS))


In [ ]:
# Cell 3 · The path-planning engine and the map app (browser code). Do not edit. (v4: diverse routes, radiation layer, best-route-per-layer)
ENGINE_JS = r'''/* ARES engine (JS port of the tested Python v3 engine). Works in the browser and in node. */
const ARES=(function(){
const NB8=[[-1,-1],[-1,0],[-1,1],[0,-1],[0,1],[1,-1],[1,0],[1,1]];
const TK=3.5,TOFF=0.05,TVMAX=6.0;
const ALGOS=["Dijkstra","A*","Weighted A*","Greedy","A* + smoothing"];
const OPT_NOTE={"Dijkstra":"guaranteed cheapest for these prices","A*":"guaranteed cheapest for these prices","Weighted A*":"close to cheapest (within the strength factor), but faster","Greedy":"very fast, NO guarantee it is cheap","A* + smoothing":"A* path shortened with straight lines that cost no more; off-grid, usually the cheapest",
  "A* via waypoint":"two A* legs forced through a waypoint to the side of the straight line, so the route is deliberately different","A* (layer)":"A* with prices built for ONE layer only (shortest, least climb, lowest radiation ...)","A* + band":"A* with a no-go band around every earlier route, so it must take a different corridor","Weighted A* (feature)":"one fast search with prices built for ONE feature, then shortened with straight lines"};
const clip=(x,a,b)=>x<a?a:(x>b?b:x);
const rint=x=>{const f=Math.floor(x),d=x-f;return d<0.5?f:(d>0.5?f+1:(f%2===0?f:f+1))};
function dec(s,T){const b=atob(s),u=new Uint8Array(b.length);for(let i=0;i<b.length;i++)u[i]=b.charCodeAt(i);return new T(u.buffer)}
function decodeData(P){return Object.assign({},P,{Z:dec(P.Z,Float64Array),slope:dec(P.slope,Float32Array),rough:dec(P.rough,Float32Array),hill:dec(P.hill,Float32Array),sand:dec(P.sand,Uint8Array)})}
const fmtHm=m=>{m=Math.round(m);const h=Math.floor(m/60),mm=m%60;return h?`${h} h ${String(mm).padStart(2,'0')} min`:`${mm} min`};
const cum=a=>{const o=new Float64Array(a.length+1);for(let i=0;i<a.length;i++)o[i+1]=o[i]+a[i];return o};
const sum=a=>{let s=0;for(let i=0;i<a.length;i++)s+=a[i];return s};
const mx=a=>{let m=-Infinity;for(let i=0;i<a.length;i++)if(a[i]>m)m=a[i];return m};
const mn=a=>{let m=Infinity;for(let i=0;i<a.length;i++)if(a[i]<m)m=a[i];return m};
const argmax=a=>{let b=0;for(let i=1;i<a.length;i++)if(a[i]>a[b])b=i;return b};
const argmin=a=>{let b=0;for(let i=1;i<a.length;i++)if(a[i]<a[b])b=i;return b};

/* ---------- exact euclidean distance transform (Felzenszwalb) ---------- */
function edt(feature,H,W){ // distance (cells) of every cell to the nearest feature cell
  const INF=1e20,f=new Float64Array(H*W);for(let i=0;i<H*W;i++)f[i]=feature[i]?0:INF;
  const n=Math.max(H,W),v=new Int32Array(n),z=new Float64Array(n+1),d=new Float64Array(n),col=new Float64Array(n);
  const pass=(get,set,len)=>{let k=0;v[0]=0;z[0]=-INF;z[1]=INF;
    for(let q=1;q<len;q++){let s=((get(q)+q*q)-(get(v[k])+v[k]*v[k]))/(2*q-2*v[k]);
      while(s<=z[k]){k--;s=((get(q)+q*q)-(get(v[k])+v[k]*v[k]))/(2*q-2*v[k])}
      k++;v[k]=q;z[k]=s;z[k+1]=INF}
    k=0;for(let q=0;q<len;q++){while(z[k+1]<q)k++;d[q]=(q-v[k])*(q-v[k])+get(v[k])}
    for(let q=0;q<len;q++)set(q,d[q])};
  for(let c=0;c<W;c++)pass(q=>f[q*W+c],(q,x)=>{f[q*W+c]=x},H);
  for(let r=0;r<H;r++)pass(q=>f[r*W+q],(q,x)=>{f[r*W+q]=x},W);
  const o=new Float64Array(H*W);for(let i=0;i<H*W;i++)o[i]=Math.sqrt(f[i]);return o}

/* ---------- radiation layer (PROXY, relative): terrain shielding + height + optional circular zones ---------- */
function skyView(D){if(D._svf)return D._svf;const H=D.H,W=D.W,px=D.px,Z=D.Z,svf=new Float32Array(H*W),steps=[1,2,3,5,8,12,18,26,36,50];
  for(let r=0;r<H;r++)for(let c=0;c<W;c++){const z0=Z[r*W+c];let s=0;
    for(let k=0;k<8;k++){const dr=NB8[k][0],dc=NB8[k][1],f=Math.hypot(dr,dc);let best=0;
      for(const st of steps){const rr=r+dr*st,cc=c+dc*st;if(rr<0||cc<0||rr>=H||cc>=W)break;const t=(Z[rr*W+cc]-z0)/(st*px*f);if(t>best)best=t}
      s+=Math.sin(Math.atan(best))}
    svf[r*W+c]=1-s/8}
  D._svf=svf;return svf}
function radLayer(D,g,TR,zones){const N=g.N,svf=skyView(D),shield=clip((TR.rad_shield_pct??50)/100,0,0.95),alt=(TR.rad_alt_pct_km??5)/100,zs=Float64Array.from(g.Z).sort(),zmed=zs[zs.length>>1];
  const base=new Float64Array(N),zm=new Float64Array(N).fill(1),noGo=new Uint8Array(N),a=g.area;
  for(let i=0;i<N;i++)base[i]=(1-shield*(1-svf[i]))*Math.max(0.2,1+alt*(g.Z[i]-zmed)/1000);
  (zones||[]).forEach(zn=>{const fr=(a.lat_max-zn.lat)/(a.lat_max-a.lat_min)*g.H-0.5,fc=(zn.lon-a.lon_min)/(a.lon_max-a.lon_min)*g.W-0.5,R=zn.radius_m,Rc=R*1.25/g.px;
    for(let rr=Math.max(0,Math.floor(fr-Rc));rr<=Math.min(g.H-1,Math.ceil(fr+Rc));rr++)for(let cc=Math.max(0,Math.floor(fc-Rc));cc<=Math.min(g.W-1,Math.ceil(fc+Rc));cc++){
      const d=g.px*Math.hypot(rr-fr,cc-fc);let v=1;if(d<=R)v=zn.mult;else if(d<=R*1.25)v=1+(zn.mult-1)*(1-(d-R)/(0.25*R));else continue;
      const i=rr*g.W+cc;if(v>zm[i])zm[i]=v;if(zn.nogo&&d<=R)noGo[i]=1}});
  const rad=new Float64Array(N);let mn_=Infinity,mx_=-Infinity;for(let i=0;i<N;i++){rad[i]=base[i]*zm[i];if(rad[i]<mn_)mn_=rad[i];if(rad[i]>mx_)mx_=rad[i]}
  return {rad,min:mn_,den:Math.max(mx_-mn_,0.5),noGo}}

/* ---------- layers that depend on the terrain rules ---------- */
function makeGrid(D,TR,zones){
  const H=D.H,W=D.W,N=H*W,px=D.px,g={H,W,N,px,Z:D.Z,slope:D.slope,rough:D.rough,sand:D.sand,hill:D.hill,hasSand:D.hasSand,area:D.area};
  g.shadow=new Uint8Array(N);g.danger=new Uint8Array(N);let any=false;
  for(let i=0;i<N;i++){g.shadow[i]=g.hill[i]<TR.shadow_level?1:0;if(g.slope[i]>TR.max_slope_deg){g.danger[i]=1;any=true}}
  g.clearance=new Float64Array(N);
  if(any){const e=edt(g.danger,H,W);for(let i=0;i<N;i++)g.clearance[i]=e[i]*px}else g.clearance.fill(1e4);
  const pct=(arr,q)=>{const a=Float64Array.from(arr).sort();return a[Math.min(a.length-1,Math.floor(q*a.length))]};
  const adapt=(TR.adaptive_contrast??1)>0;g.slopeRef=adapt?clip(pct(g.slope,.95)*1.15,3,TR.max_slope_deg):TR.max_slope_deg;g.roughRef=adapt?clip(pct(g.rough,.95)*1.15,0.05,TR.rough_threshold_m):TR.rough_threshold_m;
  const rl=radLayer(D,g,TR,zones);g.rad=rl.rad;g.radMin=rl.min;g.radDen=rl.den;
  const buf=TR.danger_buffer_m;g.blocked=new Uint8Array(N);g.closeness=new Float64Array(N);g.influence=Math.max(5*buf,2*px,1);
  for(let i=0;i<N;i++){g.blocked[i]=(g.danger[i]||g.clearance[i]<buf||rl.noGo[i])?1:0;g.closeness[i]=clip(1-g.clearance[i]/g.influence,0,1)}
  return g}
const rcToLonLat=(g,r,c)=>{const a=g.area;return [a.lon_min+(c+0.5)/g.W*(a.lon_max-a.lon_min),a.lat_max-(r+0.5)/g.H*(a.lat_max-a.lat_min)]};
const lonLatToRc=(g,lon,lat)=>{const a=g.area;return [clip(Math.round((a.lat_max-lat)/(a.lat_max-a.lat_min)*g.H-0.5),0,g.H-1),clip(Math.round((lon-a.lon_min)/(a.lon_max-a.lon_min)*g.W-0.5),0,g.W-1)]};

/* ---------- price of the ground ---------- */
function buildM(g,TR,w){const M=new Float64Array(g.N).fill(1);
  for(let i=0;i<g.N;i++){let m=1;
    if(w.slope)m+=w.slope*(TR.slope_penalty??1)*Math.pow(clip(g.slope[i]/g.slopeRef,0,1),2);
    if(w.roughness)m+=w.roughness*clip(g.rough[i]/g.roughRef,0,1);
    if(w.sand)m+=w.sand*g.sand[i];
    if(w.near_danger)m+=w.near_danger*g.closeness[i];
    if(w.height&&g.zBase!==undefined)m+=w.height*(TR.height_penalty??0)*clip((g.Z[i]-g.zBase)/g.zSpan,0,1);
    if(w.radiation)m+=w.radiation*clip((g.rad[i]-g.radMin)/g.radDen,0,1);
    M[i]=m}
  return M}

/* ---------- binary heap ordered by (priority, index) like Python's heapq on tuples ---------- */
class Heap{constructor(){this.p=[];this.v=[]}
  less(a,b){return this.p[a]<this.p[b]||(this.p[a]===this.p[b]&&this.v[a]<this.v[b])}
  swap(a,b){[this.p[a],this.p[b]]=[this.p[b],this.p[a]];[this.v[a],this.v[b]]=[this.v[b],this.v[a]]}
  push(pri,v){this.p.push(pri);this.v.push(v);let i=this.p.length-1;while(i>0){const q=(i-1)>>1;if(this.less(i,q)){this.swap(i,q);i=q}else break}}
  pop(){const top=[this.p[0],this.v[0]],lp=this.p.pop(),lv=this.v.pop();if(this.p.length){this.p[0]=lp;this.v[0]=lv;let i=0,n=this.p.length;for(;;){let l=2*i+1,r=l+1,s=i;if(l<n&&this.less(l,s))s=l;if(r<n&&this.less(r,s))s=r;if(s===i)break;this.swap(i,s);i=s}}return top}
  get size(){return this.p.length}}

function runSearch(g,M,blocked,start,goal,algo,wClimb,eps){
  const H=g.H,W=g.W,Z=g.Z,px=g.px,s=start[0]*W+start[1],t=goal[0]*W+goal[1],gr=goal[0],gc=goal[1];
  const gs=new Float64Array(g.N).fill(Infinity),par=new Int32Array(g.N).fill(-2),closed=new Uint8Array(g.N);
  gs[s]=0;par[s]=-1;const heap=new Heap();heap.push(0,s);let expanded=0;const t0=Date.now();
  while(heap.size){const [,u]=heap.pop();if(closed[u])continue;closed[u]=1;expanded++;if(u===t)break;
    const ur=(u/W)|0,uc=u-ur*W,gu=gs[u],zu=Z[u],mu=M[u];
    for(let k=0;k<8;k++){const dr=NB8[k][0],dc=NB8[k][1],rr=ur+dr,cc=uc+dc;if(rr<0||cc<0||rr>=H||cc>=W)continue;
      const v=rr*W+cc;if(blocked[v]||closed[v])continue;
      const dh=px*Math.hypot(dr,dc),dz=Z[v]-zu,ng=gu+Math.sqrt(dh*dh+dz*dz)*0.5*(mu+M[v])+(dz>0?wClimb*dz:0);
      if(ng<gs[v]){gs[v]=ng;par[v]=u;const h=px*Math.hypot(rr-gr,cc-gc);
        const pri=algo==="Dijkstra"?ng:(algo==="Greedy"?h:(algo==="Weighted A*"?ng+eps*h:ng+h));heap.push(pri,v)}}}
  if(par[t]===-2)return null;
  const path=[];for(let u=t;u!==-1;u=par[u])path.push([(u/W)|0,u%W]);path.reverse();
  return {path,cost:gs[t],expanded,secs:(Date.now()-t0)/1000}}

function bil(arr,W,H,r,c){r=clip(r,0,H-1);c=clip(c,0,W-1);const r0=Math.floor(r),c0=Math.floor(c),r1=Math.min(r0+1,H-1),c1=Math.min(c0+1,W-1),fr=r-r0,fc=c-c0;
  return arr[r0*W+c0]*(1-fr)*(1-fc)+arr[r0*W+c1]*(1-fr)*fc+arr[r1*W+c0]*fr*(1-fc)+arr[r1*W+c1]*fr*fc}
const nn=(arr,W,H,r,c)=>arr[clip(rint(r),0,H-1)*W+clip(rint(c),0,W-1)];
function lineCost(g,M,blocked,a,b,wClimb){const L=Math.hypot(b[0]-a[0],b[1]-a[1]),n=Math.max(1,Math.ceil(L/0.5)),r=[],c=[];
  for(let i=0;i<=n;i++){const t=i/n;r.push(a[0]+(b[0]-a[0])*t);c.push(a[1]+(b[1]-a[1])*t);
    if(blocked&&blocked[clip(rint(r[i]),0,g.H-1)*g.W+clip(rint(c[i]),0,g.W-1)])return null}
  const z=r.map((x,i)=>bil(g.Z,g.W,g.H,x,c[i])),m=r.map((x,i)=>bil(M,g.W,g.H,x,c[i])),dh=L*g.px/n;let tot=0,climb=0;
  for(let i=0;i<n;i++){const dz=z[i+1]-z[i];tot+=Math.hypot(dh,dz)*0.5*(m[i]+m[i+1]);if(dz>0)climb+=dz}
  return tot+wClimb*climb}
function smoothPath(g,path,M,blocked,wClimb,win=80){const n=path.length,cm=[0];
  for(let i=0;i<n-1;i++)cm.push(cm[i]+lineCost(g,M,null,path[i],path[i+1],wClimb));
  const out=[path[0]];let i=0;
  while(i<n-1){let ch=i+1;for(let j=Math.min(n-1,i+win);j>i+1;j--){const lc=lineCost(g,M,blocked,path[i],path[j],wClimb);if(lc!==null&&lc<=cm[j]-cm[i]+1e-9){ch=j;break}}out.push(path[ch]);i=ch}
  return out}
function freeCell(g,blocked,rc,radius=2){const [r,c]=rc;if(!blocked[r*g.W+c])return [rc,false];let best=null,bd=1e9;
  for(let rr=Math.max(0,r-radius);rr<Math.min(g.H,r+radius+1);rr++)for(let cc=Math.max(0,c-radius);cc<Math.min(g.W,c+radius+1);cc++)
    if(!blocked[rr*g.W+cc]&&Math.hypot(rr-r,cc-c)<bd){best=[rr,cc];bd=Math.hypot(rr-r,cc-c)}
  if(!best)throw new Error("START or END is on impassable ground and no free square is within 2 cells. Pick another point.");return [best,true]}

function cellsOf(path){const [r,c]=densify(path,1);return r.map((x,i)=>[x,c[i]])}
function pathSep(a,b){const A=cellsOf(a),B=cellsOf(b),one=(X,Y)=>{let t=0;for(const p of X){let m=1e18;for(const q of Y){const d=(p[0]-q[0])**2+(p[1]-q[1])**2;if(d<m)m=d}t+=Math.sqrt(m)}return t/X.length};return (one(A,B)+one(B,A))/2}
function pathMaxSep(a,b){const A=cellsOf(a),B=cellsOf(b);let mx=0;for(const p of A){let m=1e18;for(const q of B){const d=(p[0]-q[0])**2+(p[1]-q[1])**2;if(d<m)m=d}mx=Math.max(mx,Math.sqrt(m))}return mx}
/* ---------- path diversity helpers ---------- */
const LAYER_STYLES={ /* one route per layer: prices built for ONE thing only (plus the distance itself) */
 "Shortest":{slope:0,roughness:0,sand:0,near_danger:0,climb:0,radiation:0},
 "Least climbing":{slope:0,roughness:0,sand:0,near_danger:0,climb:8,radiation:0,height:1.5},
 "Flattest":{slope:10,roughness:0,sand:0,near_danger:0,climb:0.5,radiation:0,height:0.5},
 "Smoothest":{slope:0,roughness:8,sand:0,near_danger:0,climb:0,radiation:0},
 "Most clearance":{slope:0.5,roughness:0,sand:0,near_danger:10,climb:0,radiation:0},
 "Least sand":{slope:0,roughness:0,sand:10,near_danger:0,climb:0,radiation:0},
 "Lowest radiation":{slope:0.5,roughness:0,sand:0,near_danger:0,climb:0,radiation:12}};
function corridorHeat(g,cands){const cnt=new Uint16Array(g.N);
  cands.filter(c=>c.id!=="REF"&&c.path.length>1).forEach(c=>{const seenC=new Set();for(const [r,q] of cellsOf(c.path)){const i=clip(Math.round(r),0,g.H-1)*g.W+clip(Math.round(q),0,g.W-1);if(!seenC.has(i)){seenC.add(i);cnt[i]++}}});
  return cnt}
/* ---------- v5: ONE route per feature (single A* each), then pick the best and keep 3 ---------- */
const LAYER_OF={"Least climbing":"height","Flattest":"slope","Smoothest":"rough","Most clearance":"hazard","Least sand":"sand","Lowest radiation":"rad"};
function planFeatures(g,TR,start,goal,PRESETS,tw){
  const notes=[];let moved;
  [start,moved]=freeCell(g,g.blocked,start);if(moved)notes.push("START was on blocked ground and was moved to the nearest free square.");
  [goal,moved]=freeCell(g,g.blocked,goal);if(moved)notes.push("END was on blocked ground and was moved to the nearest free square.");
  const straight=Math.hypot(start[0]-goal[0],start[1]-goal[1]);
  {const zs=Float64Array.from(g.Z).sort(),q=f=>zs[Math.min(zs.length-1,Math.floor(f*zs.length))];g.zBase=Math.max(g.Z[start[0]*g.W+start[1]],g.Z[goal[0]*g.W+goal[1]]);g.zSpan=Math.max(q(0.95)-q(0.05),1)*0.5}
  const specs=Object.keys(LAYER_STYLES).filter(n=>n!=="Least sand"||g.hasSand).map(n=>({feature:n,w:LAYER_STYLES[n],label:"Best for: "+n}));
  specs.push({feature:"Balanced",w:PRESETS.Balanced||LAYER_STYLES.Shortest,label:"Balanced (all features)"});
  const all=[];
  specs.forEach(sp=>{const wc=(sp.w.climb||0)*(TR.climb_penalty??1),M=buildM(g,TR,sp.w),res=runSearch(g,M,g.blocked,start,goal,"Weighted A*",wc,TR.weighted_eps);
    if(!res){notes.push(sp.label+": no path exists (blocked ground).");return}
    const t0=Date.now(),path=smoothPath(g,res.path,M,g.blocked,wc,40);
    all.push({id:"A"+(all.length+1),preset:sp.feature,algo:"Weighted A* (feature)",feature:sp.feature,label:sp.label,path,expanded:res.expanded,secs:res.secs+(Date.now()-t0)/1000,relaxed:false,also:[],weights:Object.assign({},sp.w),origin:"Feature",fam:0,famIdx:0})});
  if(!all.length)throw new Error("No path exists between START and END with these rules (blocked ground).");
  const allM=all.map(c=>measureTerrain(g,TR,c)),byId={};all.forEach(c=>byId[c.id]=c);
  const shortest=Math.min(...allM.map(m=>m.dist_m)),extra=(TR.max_extra_pct??30)/100,mainExtra=(TR.main_extra_pct??15)/100;
  const pool=allM.filter(m=>m.dist_m<=shortest*(1+extra)),mainPool=allM.filter(m=>m.dist_m<=shortest*(1+mainExtra));
  const best=scorePaths(mainPool,TERRAIN_KEYS,tw)[0].m;
  const ranked=scorePaths(pool,TERRAIN_KEYS,tw).map(r=>r.m).filter(m=>m.id!==best.id);
  const minSep=Math.max(2,straight*0.05),kept=[best],sepTo=m=>Math.min(...kept.map(k=>pathSep(byId[k.id].path,byId[m.id].path)));
  for(const m of ranked){if(kept.length>=3)break;if(sepTo(m)>=minSep)kept.push(m)}
  const spare=pool.filter(m=>!kept.includes(m));
  while(kept.length<3&&spare.length){spare.sort((a,b)=>sepTo(b)-sepTo(a));const m=spare.shift();if(sepTo(m)<1.5)break;kept.push(m)}
  if(kept.length<3)notes.push(`Only ${kept.length} clearly different route(s) exist here: the ground leaves no real choice.`);
  const keys=["risk","max_slope_deg","climb_m","pct_steep","pct_rough","pct_sand","rad_idx","dist_m"],why={};
  const lenTxt=m=>{const p=(m.dist_m/best.dist_m-1)*100;return Math.abs(p)<1?"same length":`${Math.abs(p).toFixed(0)}% ${p>0?"longer":"shorter"}`};
  const shown=kept.map((m,i)=>{const c=Object.assign({},byId[m.id],{id:"P"+(i+1),fam:i,famIdx:0,also:[]});
    if(i===0)why[c.id]=`BEST: lowest overall terrain score (built for: ${byId[m.id].feature})`;
    else{const wins=keys.filter(k=>best[k]-m[k]>Math.max(1e-9,best[k]*0.1)).slice(0,2).map(k=>`${CRITERIA[k].label.toLowerCase()} ${CRITERIA[k].fmt(m[k])} vs ${CRITERIA[k].fmt(best[k])}`);
      why[c.id]=(wins.length?"ALTERNATIVE ("+byId[m.id].feature+"): better "+wins.join(", "):"BACKUP ("+byId[m.id].feature+"): different corridor in case the best route is blocked")+"; "+lenTxt(m)}
    return c});
  const byLayer={};allM.forEach(m=>{const l=LAYER_OF[byId[m.id].feature];if(l)byLayer[l]=m.id});
  return {cands:shown,all,allM,why,notes,byLayer,generated:all.length,start,goal,straight}}

/* ---------- walking helpers ---------- */
const tobler=(grade,vmax=TVMAX)=>vmax*Math.exp(-TK*Math.abs(grade+TOFF))/3.6;
function densify(path,maxStep=1.5){const rr=[path[0][0]],cc=[path[0][1]];
  for(let i=0;i<path.length-1;i++){const [r0,c0]=path[i],[r1,c1]=path[i+1],n=Math.max(1,Math.ceil(Math.hypot(r1-r0,c1-c0)/maxStep-1e-9));
    for(let k=1;k<=n;k++){rr.push(r0+(r1-r0)*k/n);cc.push(c0+(c1-c0)*k/n)}}
  return [rr,cc]}
function pathProfile(g,cand){const [r,c]=densify(cand.path),n=r.length,z=r.map((x,i)=>bil(g.Z,g.W,g.H,x,c[i])),dh=[],dz=[],sh=[],sa=[],ro=[],cl=[],ra=[];
  for(let i=0;i<n-1;i++){dh.push(Math.max(Math.hypot(r[i+1]-r[i],c[i+1]-c[i])*g.px,1e-6));dz.push(z[i+1]-z[i]);
    const mr=(r[i]+r[i+1])/2,mc=(c[i]+c[i+1])/2;sh.push(nn(g.shadow,g.W,g.H,mr,mc));sa.push(nn(g.sand,g.W,g.H,mr,mc));ro.push(bil(g.rough,g.W,g.H,mr,mc));cl.push(bil(g.clearance,g.W,g.H,mr,mc));ra.push(bil(g.rad,g.W,g.H,mr,mc))}
  return {r,c,z,dh,dz,shadow:sh,sand:sa,rough:ro,clear:cl,rad:ra}}

/* ---------- STAGE 1: terrain only ---------- */
function measureTerrain(g,TR,cand){
  const p=pathProfile(g,cand),n=p.dh.length,d3=p.dh.map((h,i)=>Math.hypot(h,p.dz[i])),wsum=sum(d3),cumd=cum(d3),grade=p.dh.map((h,i)=>p.dz[i]/h);
  const slope=grade.map(x=>Math.atan(Math.abs(x))*180/Math.PI),risky=TR.risky_slope_deg,imp=TR.max_slope_deg;
  const tref=d3.map((d,i)=>d/Math.max(tobler(grade[i]),0.05)/60),cumT=cum(tref);
  const ll=p.r.map((r,i)=>rcToLonLat(g,r,p.c[i])),wtxt=i=>`lat ${ll[i][1].toFixed(4)}, lon ${ll[i][0].toFixed(4)}`;
  const pct=f=>{let s=0;for(let i=0;i<n;i++)if(f(i))s+=d3[i];return s/wsum*100};
  const edges=[...new Set([0,5,10,risky,imp])].sort((a,b)=>a-b).concat([999]),bands={};
  for(let k=0;k<edges.length-1;k++){const lo=edges[k],hi=edges[k+1];bands[hi<999?`${lo}–${hi}°`:`>${lo}°`]=pct(i=>slope[i]>=lo&&slope[i]<hi)}
  const near=p.clear.map(c=>c<g.influence*0.5),steep=slope.map(s=>s>risky),roughM=p.rough.map(x=>x>TR.rough_threshold_m),sandM=p.sand.map(x=>x>0.5);
  let run=0,bestRun=0,bestStart=0,curStart=0;
  for(let i=0;i<n;i++){if(steep[i]){if(run===0)curStart=i;run+=d3[i];if(run>bestRun){bestRun=run;bestStart=curStart}}else run=0}
  const sn=slope.map(s=>clip(s/imp,0,1)),sp=0.5*sum(sn)/n+0.5*mx(sn);let dpS=0;for(let i=0;i<n;i++)dpS+=clip(1-p.clear[i]/g.influence,0,1)*d3[i];
  const risk=0.6*sp+0.4*(dpS/wsum),iSteep=argmax(slope),iClear=argmin(p.clear);
  let climb=0,descent=0;for(const x of p.dz){if(x>0)climb+=x;else descent-=x}
  const r0=p.r[0],c0=p.c[0],r1=p.r[p.r.length-1],c1=p.c[p.c.length-1],straight=Math.hypot(r1-r0,c1-c0)*g.px,detour=wsum/Math.max(straight,1e-6);
  const stype=slope.map((s,i)=>sandM[i]?"sand":(steep[i]?"steep":(roughM[i]?"rough":"firm"))),names={sand:"loose sand",steep:"steep ground",rough:"rough ground",firm:"firm ground"};
  const bounds=[];let st=0,acc=0,curT=stype[0];
  for(let i=0;i<n;i++){acc+=d3[i];if(acc>=TR.leg_length_m||(stype[i]!==curT&&acc>=100)){bounds.push([st,i]);st=i+1;acc=0;curT=i+1<n?stype[i+1]:stype[i]}}
  if(st<n)bounds.push([st,n-1]);
  const legs=bounds.map(([a,b],k)=>{let w=0,sl=0,mxs=0,cl=0,tr=0;const cnt={};for(let i=a;i<=b;i++){w+=d3[i];sl+=slope[i]*d3[i];mxs=Math.max(mxs,slope[i]);if(p.dz[i]>0)cl+=p.dz[i];tr+=tref[i];cnt[stype[i]]=(cnt[stype[i]]||0)+1}
    const top=Object.keys(cnt).sort((x,y)=>cnt[y]-cnt[x])[0];return {"Leg":k+1,"From km":cumd[a]/1000,"Length m":w,"Avg slope °":sl/w,"Max slope °":mxs,"Ground":names[top],"Climb m":cl,"Reference time min":tr}});
  const problems=[],cl0=mn(p.clear);
  if(steep.some(x=>x)){const ps=pct(i=>steep[i]);problems.push([ps>10?"HIGH":"MEDIUM",`${(ps*wsum/100).toFixed(0)} m (${ps.toFixed(0)}% of the path) is steeper than ${risky}°. Steepest: ${slope[iSteep].toFixed(1)}° at km ${(cumd[iSteep]/1000).toFixed(1)} (${wtxt(iSteep)}). Longest steep stretch: ${bestRun.toFixed(0)} m, starting at km ${(cumd[bestStart]/1000).toFixed(1)}.`,iSteep])}
  if(cl0<TR.danger_buffer_m*3+g.px)problems.push(["MEDIUM",`Passes within ${cl0.toFixed(0)} m of impassable ground at km ${(cumd[iClear]/1000).toFixed(1)} (${wtxt(iClear)}). ${pct(i=>near[i]).toFixed(0)}% of the path is in the 'close to danger' zone.`,iClear])
  if(pct(i=>roughM[i])>5)problems.push(["MEDIUM",`${pct(i=>roughM[i]).toFixed(0)}% of the path is rough (height wobble above ${TR.rough_threshold_m} m).`,roughM.indexOf(true)])
  if(g.hasSand&&sandM.some(x=>x))problems.push(["MEDIUM",`${pct(i=>sandM[i]).toFixed(0)}% of the path crosses mapped sand (first at km ${(cumd[sandM.indexOf(true)]/1000).toFixed(1)}).`,sandM.indexOf(true)])
  if(climb>0.15*wsum)problems.push(["MEDIUM",`A lot of climbing: +${climb.toFixed(0)} m up and −${descent.toFixed(0)} m down over ${(wsum/1000).toFixed(1)} km.`,argmax(p.z)])
  if(detour>1.3)problems.push(["INFO",`The path is ${detour.toFixed(2)}× the straight-line distance: it detours around hard ground.`,p.z.length>>1])
  const rsum=sum(p.rad.map((x,i)=>x*d3[i])),radIdx=rsum/wsum,hotD=sum(p.rad.map((x,i)=>x>1.25?d3[i]:0)),hotPct=hotD/wsum*100,iRad=argmax(p.rad),radMax=mx(p.rad);
  if(hotD>0)problems.push([hotPct>25?"HIGH":"MEDIUM",`${hotD.toFixed(0)} m (${hotPct.toFixed(0)}% of the path) is in elevated radiation (more than 1.25× open ground). Peak ${radMax.toFixed(1)}× at km ${(cumd[iRad]/1000).toFixed(1)} (${wtxt(iRad)}).`,iRad]);
  if(cand.relaxed)problems.push(["INFO","This route style's own slope cap could not be met, so the impassable limit was used instead.",0])
  if(cand.preset==="Curiosity reference")problems.push(["INFO","Straight lines between the typed waypoints; NOT checked against blocked ground.",0])
  const trust=[`Heights: ${g.label}`,`Sand: ${g.sandLabel}`,`Grid cell = ${g.px.toFixed(0)} m`];if(g.isDemo)trust.unshift("DEMO — synthetic terrain, results are NOT about real Mars ground.");
  (g.notes||[]).forEach(x=>trust.push(x));if(!g.hasSand)trust.push("Sand not checked: no sand mask loaded, so sand cost and sand % are zero.");
  trust.push("Radiation index is a PROXY: terrain shielding (sky view) + a small height effect + any zones you drew. 1.0 = open ground. It is not a dose model.");trust.push("Reference time = Tobler's published rule for an unburdened Earth hiker (best 6 km/h). NOT a suited-astronaut time.");
  const head=`${(wsum/1000).toFixed(2)} km · +${climb.toFixed(0)} m / −${descent.toFixed(0)} m · steepest ${mx(slope).toFixed(1)}° · ${pct(i=>steep[i]).toFixed(0)}% steep · reference walk ${fmtHm(cumT[n])}`;
  let wm=0;for(let i=0;i<n;i++)wm+=slope[i]*d3[i];
  return {id:cand.id,label:cand.label,preset:cand.preset,algo:cand.algo,also:cand.also,expanded:cand.expanded,secs:cand.secs,found_note:OPT_NOTE[cand.algo]||"typed reference points",relaxed:!!cand.relaxed,
    dist_m:wsum,straight_m:straight,rad_idx:radIdx,pct_hot:hotPct,rad_max:radMax,fam:cand.fam==null?"":"F"+(cand.fam+1),origin:cand.origin||"",detour,ref_time_min:cumT[n],climb_m:climb,descent_m:descent,zmax:mx(p.z),zmin:mn(p.z),mean_slope_deg:wm/wsum,max_slope_deg:mx(slope),
    steep_km:cumd[iSteep]/1000,steep_run_m:bestRun,closest_m:cl0,closest_km:cumd[iClear]/1000,bands,pct_steep:pct(i=>steep[i]),pct_rough:pct(i=>roughM[i]),pct_sand:pct(i=>sandM[i]),pct_near:pct(i=>near[i]),risk,
    problems,trust,head,legs,leg_bounds:bounds,hard_fail:false,fails:[],prof:p,
    arr:{lon:ll.map(x=>x[0]),lat:ll.map(x=>x[1]),z:p.z,cumd,slope,d3,steep,near}}}

/* ---------- STAGE 2: assumptions ---------- */
const minetti=g=>{g=clip(g,-0.45,0.45);return 280.5*g**5-58.7*g**4-76.8*g**3+51.9*g**2+19.6*g+2.5};
function physics(dh,dz,shadow,A,rad){const n=dh.length,o={d3:[],t_min:[],o2:[],o2_slope:[],o2_energy:[],batt:[],rad:[]},vmax=A.flat_speed_kmh/Math.exp(-TK*TOFF);
  for(let i=0;i<n;i++){const d3=Math.hypot(dh[i],dz[i]),gr=dz[i]/dh[i],v=Math.max(tobler(gr,vmax)*A.slowest_walker_factor,0.05),ts=d3/v,tm=ts/60,th=ts/3600,th_=Math.atan(gr),rest=A.o2_rest_l_min;
    const os=tm*(rest+A.o2_slope_coeff*Math.abs(Math.sin(th_))),kj=minetti(gr)*A.suit_mass_kg*d3/1000,oe=Math.max(kj/A.o2_kj_per_l,rest*tm);
    const o2=A.o2_model==="slope"?os:(A.o2_model==="energy"?oe:Math.max(os,oe)),pw=A.life_support_w+A.heater_w+A.other_w+A.cold_extra_w*shadow[i];
    o.d3.push(d3);o.t_min.push(tm);o.o2.push(o2);o.o2_slope.push(os);o.o2_energy.push(oe);o.batt.push(pw*th/1000);o.rad.push(A.rad_msv_day/24*th*(rad?rad[i]:1))}
  return o}
function measureFull(g,TR,A,m,withLegs=true){
  const p=m.prof,out=physics(p.dh,p.dz,p.shadow,A,p.rad),back=physics(p.dh,p.dz.map(x=>-x),p.shadow,A,p.rad),n=p.dh.length+1,cumd=m.arr.cumd;
  const cT=cum(out.t_min),cO=cum(out.o2),cB=cum(out.batt),cR=cum(out.rad),rT=cum(back.t_min),rO=cum(back.o2),rB=cum(back.batt),tank=A.o2_tank_l,cap=A.battery_kwh;
  const o2l=Array.from(cO,x=>A.start_o2_pct-x/tank*100),bl=Array.from(cB,x=>A.start_batt_pct-x/cap*100);
  let firstBad=-1;for(let i=0;i<n;i++){const ok=(o2l[i]-rO[i]/tank*100)>=A.o2_reserve_pct&&(bl[i]-rB[i]/cap*100)>=A.batt_reserve_pct&&(cT[i]+rT[i])<=A.eva_limit_h*60;if(!ok){firstBad=i;break}}
  const pnrNone=firstBad===0,pnrIdx=pnrNone?0:(firstBad>0?firstBad-1:n-1),fullOk=firstBad===-1,imp=TR.max_slope_deg,last=n-1,fails=[];
  if(o2l[last]<0)fails.push("CRITICAL: INSUFFICIENT OXYGEN (tank runs out before the end)");else if(o2l[last]<A.o2_reserve_pct)fails.push(`oxygen left ${o2l[last].toFixed(0)}% is under the ${A.o2_reserve_pct}% reserve`);
  if(bl[last]<A.batt_reserve_pct)fails.push(`battery left ${bl[last].toFixed(0)}% is under the ${A.batt_reserve_pct}% reserve`);
  if(cT[last]>A.eva_limit_h*60)fails.push(`walk time ${fmtHm(cT[last])} is over the ${A.eva_limit_h} h EVA limit`);
  if(mx(m.arr.slope)>imp+0.5)fails.push(`a step is ${mx(m.arr.slope).toFixed(1)}° (impassable limit ${imp}°)`);
  const warns=[],used=o2l.map(x=>A.start_o2_pct-x);
  if(o2l[last]<0)warns.push(["CRITICAL WARNING: INSUFFICIENT OXYGEN",o2l.findIndex(x=>x<0)]);
  let i=used.findIndex(x=>x>=A.o2_warn_pct);if(i>=0)warns.push([`Oxygen passes ${A.o2_warn_pct}% used at ${(cumd[i]/1000).toFixed(1)} km`,i]);
  i=bl.findIndex(x=>x<A.batt_reserve_pct);if(i>=0)warns.push([`Battery drops under ${A.batt_reserve_pct}% at ${(cumd[i]/1000).toFixed(1)} km`,i]);
  if(A.rad_warn_msv>0&&cR[last]>=A.rad_warn_msv)warns.push([`Radiation dose ${cR[last].toFixed(3)} mSv reaches the warning level`,last]);
  if(pnrNone)warns.push(["Even the first step leaves too little reserve to walk back",0]);else if(!fullOk)warns.push([`Point of no return at ${(cumd[pnrIdx]/1000).toFixed(1)} km — turn back before this`,pnrIdx]);
  let legs_full=null;if(withLegs)legs_full=m.leg_bounds.map(([a,b],k)=>Object.assign({},m.legs[k],{"Time so far":fmtHm(cT[b+1]),"Oxygen left %":o2l[b+1],"Battery left %":bl[b+1]}));
  if(legs_full)legs_full.forEach(r=>delete r["Reference time min"]);
  const ro2=Array.from(rO,x=>x/tank*100),rb=Array.from(rB,x=>x/cap*100);
  return {time_min:cT[last],o2_used_l:cO[last],o2_used_pct:A.start_o2_pct-o2l[last],o2_left_pct:o2l[last],o2_slope_l:sum(out.o2_slope),o2_energy_l:sum(out.o2_energy),
    batt_used_kwh:cB[last],batt_used_pct:A.start_batt_pct-bl[last],batt_left_pct:bl[last],rad_msv:cR[last],hard_fail:fails.length>0,fails,warns,
    trust_full:[`Oxygen formula used: ${A.o2_model} (energy formula is Earth-based).`,"Speed, oxygen, battery, radiation and reserves are ASSUMPTIONS (page 3).","Radiation dose = dose rate on open ground × the radiation index along the route (a proxy, not a dose model).",A.cold_extra_w>0?"Shadow is only a proxy from the height map + sun angle.":"Shadow extra heating is off (0 W)."],
    rt_time_min:cT[last]+rT[last],rt_o2_pct:A.start_o2_pct-(o2l[last]-ro2[last]),rt_batt_pct:A.start_batt_pct-(bl[last]-rb[last]),pnr_km:cumd[pnrIdx]/1000,pnr_idx:pnrIdx,pnr_none:pnrNone,full_return_ok:fullOk,legs_full,
    arr_full:{cumt:cT,o2_left:o2l,b_left:bl,ret_o2:ro2,ret_b:rb}}}
const runStage2=(g,TR,A,tms,legs=true)=>tms.map(m=>Object.assign({},m,measureFull(g,TR,A,m,legs)));

/* ---------- the score system ---------- */
const CRITERIA={dist_m:{label:"Distance",fmt:v=>(v/1000).toFixed(2)+" km",stage:"both"},ref_time_min:{label:"Reference time",fmt:fmtHm,stage:"terrain"},time_min:{label:"Walk time",fmt:fmtHm,stage:"full"},
  climb_m:{label:"Total climb",fmt:v=>v.toFixed(0)+" m",stage:"both"},mean_slope_deg:{label:"Average slope",fmt:v=>v.toFixed(1)+"°",stage:"both"},max_slope_deg:{label:"Steepest spot",fmt:v=>v.toFixed(1)+"°",stage:"both"},
  pct_steep:{label:"Steep ground",fmt:v=>v.toFixed(0)+"%",stage:"both"},pct_rough:{label:"Rough ground",fmt:v=>v.toFixed(0)+"%",stage:"both"},pct_sand:{label:"Sand",fmt:v=>v.toFixed(0)+"%",stage:"both"},
  pct_near:{label:"Near cliffs",fmt:v=>v.toFixed(0)+"%",stage:"both"},rad_idx:{label:"Radiation exposure",fmt:v=>v.toFixed(2)+"×",stage:"both"},risk:{label:"Risk index",fmt:v=>v.toFixed(2),stage:"both"},o2_used_pct:{label:"Oxygen used",fmt:v=>v.toFixed(0)+"%",stage:"full"},
  batt_used_pct:{label:"Battery used",fmt:v=>v.toFixed(0)+"%",stage:"full"},rad_msv:{label:"Radiation dose",fmt:v=>v.toFixed(3)+" mSv",stage:"full"}};
const TERRAIN_KEYS=Object.keys(CRITERIA).filter(k=>CRITERIA[k].stage!=="full"),FULL_KEYS=Object.keys(CRITERIA).filter(k=>CRITERIA[k].stage!=="terrain");
const TERRAIN_PRIORITIES={"Balanced":{dist_m:.08,ref_time_min:.08,climb_m:.25,mean_slope_deg:.15,max_slope_deg:.12,pct_steep:.10,pct_rough:.05,pct_sand:.05,pct_near:.05,rad_idx:.05,risk:.15},
  "Safest ground":{risk:.30,max_slope_deg:.20,pct_steep:.15,pct_near:.10,pct_rough:.05,pct_sand:.05,mean_slope_deg:.05,dist_m:.05,ref_time_min:.05},
  "Shortest / fastest":{dist_m:.40,ref_time_min:.40,climb_m:.05,risk:.10,mean_slope_deg:.05},"Least climbing":{climb_m:.40,mean_slope_deg:.20,dist_m:.15,ref_time_min:.10,risk:.10,max_slope_deg:.05},"Lowest radiation":{rad_idx:.60,risk:.10,dist_m:.10,ref_time_min:.10,max_slope_deg:.10}};
const FULL_PRIORITIES={"Balanced":{risk:.25,time_min:.20,o2_used_pct:.15,batt_used_pct:.15,dist_m:.10,climb_m:.05,rad_msv:.05,pct_steep:.05},
  "Safety first":{risk:.45,max_slope_deg:.10,time_min:.10,o2_used_pct:.10,batt_used_pct:.10,pct_steep:.10,dist_m:.05},"Speed first":{time_min:.50,dist_m:.15,risk:.15,o2_used_pct:.10,batt_used_pct:.10},
  "Energy / oxygen first":{o2_used_pct:.30,batt_used_pct:.25,climb_m:.15,time_min:.10,risk:.10,dist_m:.10},"Lowest radiation":{rad_msv:.45,rad_idx:.25,risk:.10,time_min:.10,dist_m:.10}};
function scorePaths(ms,crit,weights){const w={};let tot=0;crit.forEach(k=>{w[k]=Math.max(+weights[k]||0,0);tot+=w[k]});if(tot<=0){crit.forEach(k=>w[k]=1);tot=crit.length}
  let pool=ms.filter(m=>!m.hard_fail);if(!pool.length)pool=ms;const lo={},hi={};crit.forEach(k=>{lo[k]=Math.min(...pool.map(m=>m[k]));hi[k]=Math.max(...pool.map(m=>m[k]))});
  const out=pool.map(m=>{const r={m,id:m.id,label:m.label,n:{},c:{},score:0};crit.forEach(k=>{const rg=hi[k]-lo[k];r.n[k]=rg>0?(m[k]-lo[k])/rg:0;r.c[k]=r.n[k]*w[k]/tot;r.score+=r.c[k]});return r});
  out.sort((a,b)=>a.score-b.score);out.forEach((r,i)=>r.rank=i+1);return out}
function explainRanking(ranked,crit,weights){const n=ranked.length,active=crit.filter(k=>(weights[k]||0)>0),act=active.length?active:crit,pos={};
  crit.forEach(k=>{const vals=ranked.map(r=>r.m[k]);pos[k]={};ranked.forEach(r=>{pos[k][r.id]=1+vals.filter(v=>v<r.m[k]).length})});
  const strengths=ranked.map(r=>{const gd=[...act].sort((a,b)=>pos[a][r.id]-pos[b][r.id]||(a<b?-1:1)).slice(0,2),bd=[...act].sort((a,b)=>pos[b][r.id]-pos[a][r.id]||(a<b?-1:1)).slice(0,2);
    return {id:r.id,Rank:r.rank,Strengths:gd.map(k=>(pos[k][r.id]===1?"best ":"")+`${CRITERIA[k].label.toLowerCase()} (#${pos[k][r.id]}/${n})`).join(", "),Weaknesses:bd.map(k=>(pos[k][r.id]===n?"worst ":"")+`${CRITERIA[k].label.toLowerCase()} (#${pos[k][r.id]}/${n})`).join(", ")}});
  const gap=ranked.map(r=>{const o={id:r.id,Rank:r.rank};act.forEach(k=>{const best=Math.min(...ranked.map(x=>x.m[k])),d=r.m[k]-best;o[CRITERIA[k].label+" vs best"]=d<=1e-9?"best":(k==="time_min"||k==="ref_time_min"?`+${d.toFixed(0)} min`:"+"+CRITERIA[k].fmt(d))});return o});
  const winners={};act.forEach(k=>{const best=Math.min(...ranked.map(x=>x.m[k]));winners[CRITERIA[k].label]=ranked.filter(r=>r.m[k]<=best+1e-9).map(r=>r.id).join(", ")+`  (${CRITERIA[k].fmt(best)})`});
  const pareto=ranked.filter((r,i)=>!ranked.some((s,j)=>j!==i&&act.every(k=>s.m[k]<=r.m[k]+1e-12)&&act.some(k=>s.m[k]<r.m[k]-1e-12))).map(r=>r.id);
  const seen={};ranked.forEach(r=>seen[r.id]=[r.rank]);const ms=ranked.map(r=>r.m);
  act.forEach(k=>[0.8,1.2].forEach(f=>{const w2=Object.assign({},weights);w2[k]=(w2[k]||0)*f;scorePaths(ms,crit,w2).forEach(r=>seen[r.id].push(r.rank))}));
  const stability={};for(const id in seen){const a=seen[id],lo=Math.min(...a),hi=Math.max(...a);stability[id]=lo===hi?"firm":`flips (ranks ${lo}–${hi})`}
  return {strengths,gap,winners,pareto,stability}}
function formulaText(crit,weights){const tot=crit.reduce((s,k)=>s+Math.max(weights[k]||0,0),0)||1;
  return "score = "+crit.filter(k=>(weights[k]||0)>0).map(k=>`${(Math.max(weights[k],0)/tot).toFixed(2)} × ${CRITERIA[k].label}`).join(" + ")+"   (each criterion scaled 0 = best path … 1 = worst; lower score wins)"}

/* ---------- which assumptions matter? ---------- */
const INFL=[["time_min","Time"],["o2_used_pct","Oxygen"],["batt_used_pct","Battery"],["rad_msv","Radiation"]];
function assumptionInfluence(g,TR,A,A_SPEC,tms,weights,delta=0.25,progress){
  const base=runStage2(g,TR,A,tms,false),bwin=scorePaths(base,FULL_KEYS,weights)[0].id,bfails=new Set(base.filter(m=>m.hard_fail).map(m=>m.id)),rows=[];
  const keys=A_SPEC.filter(s=>(s.kind==="num"&&!s.info)||s.key==="o2_model");
  keys.forEach((s,ki)=>{const key=s.key;if(progress)progress(`testing ${s.label} (${ki+1}/${keys.length})`);let trials=[];
    if(s.kind==="choice")trials=s.options.filter(o=>o!==A[key]);else{const v0=A[key];
      if(v0===0){rows.push({key,Assumption:s.label,Tag:s.tag,Now:0,Unit:s.unit,Affects:"— (value is 0: nothing to scale)",Time:0,Oxygen:0,Battery:0,Radiation:0,Flips:0,WinnerChanges:"no",Impact:"none",_score:0});return}
      [1-delta,1+delta].forEach(f=>{let v=v0*f;if(s.lo!=null)v=Math.max(v,s.lo);if(s.hi!=null)v=Math.min(v,s.hi);trials.push(v)})}
    const best={};INFL.forEach(([k])=>best[k]=0);let flips=0,win=false;
    trials.forEach(v=>{const A2=Object.assign({},A,{[key]:v}),alt=runStage2(g,TR,A2,tms,false);
      INFL.forEach(([mk])=>{let s_=0;alt.forEach((x,i)=>s_+=Math.abs(x[mk]-base[i][mk])/Math.max(Math.abs(base[i][mk]),1e-9));best[mk]=Math.max(best[mk],s_/alt.length*100)});
      const af=new Set(alt.filter(m=>m.hard_fail).map(m=>m.id));flips=Math.max(flips,[...bfails].filter(x=>!af.has(x)).length+[...af].filter(x=>!bfails.has(x)).length);
      win=win||scorePaths(alt,FULL_KEYS,weights)[0].id!==bwin});
    const top=Math.max(...Object.values(best)),impact=(win||flips>0||top>=15)?"HIGH":(top>=5?"MEDIUM":(top>=0.5?"LOW":"none")),aff=INFL.filter(([mk])=>best[mk]>=0.5).map(x=>x[1]);if(flips>0)aff.push(`pass/fail of up to ${flips} paths`);
    rows.push({key,Assumption:s.label,Tag:s.tag,Now:A[key],Unit:s.unit,Affects:aff.join(", ")||"nothing measurable",Time:best.time_min,Oxygen:best.o2_used_pct,Battery:best.batt_used_pct,Radiation:best.rad_msv,Flips:flips,WinnerChanges:win?"YES":"no",Impact:impact,_score:top+100*flips+1000*(win?1:0)})});
  return rows.sort((a,b)=>b._score-a._score)}
function plainSummary(tms,fms,infl,tw,fw){const lines=[],lab={};tms.forEach(m=>lab[m.id]=m.label);
  const tb=scorePaths(tms,TERRAIN_KEYS,tw)[0].id,fb=scorePaths(fms,FULL_KEYS,fw)[0].id;
  lines.push(`Terrain only, the best path is ${tb} (${lab[tb]}).`);lines.push(`With the assumptions added, the best path is ${fb} (${lab[fb]})`+(fb!==tb?" — the winner changed.":" — the same winner."));
  const nf=fms.filter(m=>m.hard_fail).length;
  if(nf){const rs={};fms.forEach(m=>m.fails.forEach(f=>{const k=f.includes("battery")?"battery reserve":(f.toLowerCase().includes("oxygen")?"oxygen":(f.includes("EVA")?"EVA time limit":"slope"));rs[k]=(rs[k]||0)+1}));
    lines.push(`${nf} of ${fms.length} paths break a hard rule once assumptions are added (${Object.entries(rs).sort((a,b)=>b[1]-a[1]).map(([k,v])=>`${v}× ${k}`).join(", ")}). They look fine on terrain alone.`)}else lines.push("No path breaks a hard rule with the current assumptions.");
  if(infl&&infl.length){const top=infl.filter(r=>r.Impact==="HIGH").slice(0,3);lines.push(top.length?"Assumptions that matter most: "+top.map(r=>r.Assumption).join(", ")+".":"No single assumption (±25%) changes a pass/fail result or the winner.")}
  lines.push(`On average the suited walk takes ${(fms.reduce((s,m)=>s+m.time_min/m.ref_time_min,0)/fms.length).toFixed(1)}× the standard-hiker reference time.`);return lines}

return {pathSep,pathMaxSep,ALGOS,decodeData,makeGrid,rcToLonLat,lonLatToRc,buildM,planFeatures,measureTerrain,measureFull,runStage2,CRITERIA,TERRAIN_KEYS,FULL_KEYS,TERRAIN_PRIORITIES,FULL_PRIORITIES,
  scorePaths,explainRanking,formulaText,assumptionInfluence,plainSummary,fmtHm,freeCell,edt,corridorHeat,LAYER_STYLES,skyView}})();
if(typeof module!=='undefined')module.exports=ARES;
'''
UI_HTML = r'''<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><title>ARES Mars Route Planner v5</title>
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<style>
:root{--bg:#fff;--fg:#1b1f23;--mut:#5b6570;--card:#f4f6f8;--line:#d6dbe0;--acc:#c1440e;--ok:#1a7f37;--bad:#c62828;--warn:#b26a00;box-sizing:border-box;padding-top:env(safe-area-inset-top,0px);padding-bottom:env(safe-area-inset-bottom,0px)}
@media(prefers-color-scheme:dark){:root:not([data-theme="light"]){--bg:#14171a;--fg:#e8ebee;--mut:#9aa5b0;--card:#1e2327;--line:#343b42;--acc:#ff7a45;--ok:#4cc26a;--bad:#ff6b6b;--warn:#f0b050}}
:root[data-theme="dark"]{--bg:#14171a;--fg:#e8ebee;--mut:#9aa5b0;--card:#1e2327;--line:#343b42;--acc:#ff7a45;--ok:#4cc26a;--bad:#ff6b6b;--warn:#f0b050}
html{scroll-padding-top:env(safe-area-inset-top,0px)}
body{margin:0;background:var(--bg);color:var(--fg);font:14px/1.45 system-ui,Arial,sans-serif}
.w{max-width:1100px;margin:0 auto;padding:12px}
h1{font-size:20px;margin:4px 0}h2{font-size:16px;margin:14px 0 6px}
.demo{background:var(--warn);color:#000;padding:4px 10px;border-radius:6px;font-weight:600;display:inline-block;margin:4px 0}
nav{display:flex;gap:4px;overflow-x:auto;position:sticky;top:env(safe-area-inset-top,0px);background:var(--bg);padding:6px 0;z-index:5}
nav button,.btn{background:var(--card);color:var(--fg);border:1px solid var(--line);border-radius:6px;padding:6px 10px;cursor:pointer;white-space:nowrap;font:inherit}
nav button.on,.btn.pri{background:var(--acc);color:#fff;border-color:var(--acc)}
.card{background:var(--card);border:1px solid var(--line);border-radius:8px;padding:10px;margin:8px 0}
.row{display:flex;gap:12px;flex-wrap:wrap}.row>*{flex:1 1 300px;min-width:0}
.sc{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:12.5px}th,td{border-bottom:1px solid var(--line);padding:4px 6px;text-align:left;white-space:nowrap}th{color:var(--mut)}
tr.sel td{background:rgba(193,68,14,.15)}tr.cl{cursor:pointer}
.ok{color:var(--ok)}.bad{color:var(--bad)}.wn{color:var(--warn)}.mut{color:var(--mut)}
canvas{max-width:100%;display:block;touch-action:none;border:1px solid var(--line);border-radius:6px;cursor:crosshair;touch-action:manipulation}
label.s{display:grid;grid-template-columns:1fr 90px;gap:6px;align-items:center;margin:3px 0;font-size:13px}
input,select{background:var(--bg);color:var(--fg);border:1px solid var(--line);border-radius:4px;padding:3px 5px;font:inherit;max-width:100%}
input[type=range]{padding:0}
.mapw{position:relative}.mapw .lyr{position:absolute;top:8px;left:8px;z-index:3}.mapw .zb{position:absolute;top:8px;right:8px;display:flex;flex-direction:column;gap:4px}.mapw .cap{position:absolute;left:8px;bottom:6px;font-size:11px;color:#fff;background:rgba(0,0,0,.5);padding:1px 6px;border-radius:4px}.lpn{background:var(--card);border:1px solid var(--line);border-radius:6px;padding:6px 8px;margin-top:4px;font-size:12px;max-width:260px}.lpn label{display:block;margin:2px 0}.pg{display:none}.pg.on{display:block}
textarea{width:100%;height:160px;background:var(--bg);color:var(--fg);border:1px solid var(--line);border-radius:6px;font:12px monospace}
.leg span{display:inline-block;margin-right:10px;font-size:12px}.leg i{display:inline-block;width:12px;height:12px;vertical-align:-2px;margin-right:3px;border-radius:2px}
</style></head><body><div class="w">
<h1>🔴 ARES — Mars Route Planner v5</h1>
<div id="demo"></div>
<nav id="nav"></nav>
<div id="p1" class="pg"></div><div id="p2" class="pg"></div><div id="p3" class="pg"></div><div id="p4" class="pg"></div><div id="p5" class="pg"></div><div id="p6" class="pg"></div><div id="p7" class="pg"></div>
</div>
<script>__ENGINE__</script>
<script>
const P=__PAYLOAD__;
const $=id=>document.getElementById(id),esc=s=>String(s).replace(/[&<>]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;'}[c]));
let D=ARES.decodeData(P);const TR={},A={};P.TR_SPEC.forEach(s=>TR[s.key]=s.default);P.A_SPEC.forEach(s=>A[s.key]=s.default);
const ZONES=[];let zoneMode=false,famOnly=false,layerPick='';
let g=ARES.makeGrid(D,TR,ZONES),S=null,E=null,pl=null,tms=[],fms=[],sel=null,layer='height',infl=null,stage='full',prof={t:'Balanced',f:'Balanced'},tw={...ARES.TERRAIN_PRIORITIES.Balanced},fw={...ARES.FULL_PRIORITIES.Balanced},page=1;
const K=4,CR=ARES.CRITERIA,fmt=(k,v)=>CR[k]?CR[k].fmt(v):v;
const NAMES=['1 Pick','2 Terrain paths','3 Assumptions','4 What changed','5 Score','6 Path cards','7 Export & audit'];
const PROFILES={Balanced:'Balanced'};
$('nav').innerHTML=NAMES.map((n,i)=>`<button data-i="${i+1}">${n}</button>`).join('');
$('nav').onclick=e=>{const b=e.target.closest('button');if(b)go(+b.dataset.i)};
const banner=()=>{$('demo').innerHTML=P.isDemo?'<span class="demo">DEMO — not Mars</span> <span class="mut">'+esc(P.notes[0]||'')+'</span>':'<span class="mut">Real data: <b>'+esc(P.label)+'</b> · lat '+P.area.lat_min.toFixed(3)+' to '+P.area.lat_max.toFixed(3)+', lon '+P.area.lon_min.toFixed(3)+' to '+P.area.lon_max.toFixed(3)+' · grid cell '+Math.round(P.px)+' m · '+esc(P.sandLabel)+(P.notes.length?'<br>⚠ '+esc(P.notes.join(' ')):'')+'</span>'};banner();
function go(i){page=i;[...document.querySelectorAll('.pg')].forEach((p,j)=>p.classList.toggle('on',j+1===i));[...$('nav').children].forEach((b,j)=>b.classList.toggle('on',j+1===i));render()}
/* ---------- map: real imagery tiles + optional layers (zoom with wheel / buttons, drag to pan) ---------- */
const LAYERS=[['height','Height','dark = low, bright = high'],['slope','Slope','dark = flat, bright = steep'],['rough','Roughness','dark = smooth, bright = bumpy'],['hazard','Hazard','red blocked, orange risky, yellow rough'],['sand','Sand','light = sand mask'],['rad','Radiation (relative)','blue = shielded, red = hot zone'],['corridor','Route agreement','brighter = more routes use it'],['cost','Why: total cost','green = cheap to walk, red = expensive (all features mixed)']];
const ovl={height:false,slope:false,rough:false,hazard:false,sand:false,rad:false,corridor:false,cost:false};let ovlAlpha=0.6,lc={g:null},tileIdx=0,tilesOK=null,view=null,maps=[];
const tileCache={},TMAX=P.tileMaxZoom||7;let A0=P.area;
const ramp=t=>{t=Math.max(0,Math.min(1,t));return[Math.round(40+215*t),Math.round(30+150*t*t+40*t),Math.round(25+60*t*t)]};
const bestId=()=>{try{const r=ARES.scorePaths(tms,ARES.TERRAIN_KEYS,tw);return r.length?r[0].id:null}catch(e){return null}};
const pcol=id=>{if(id===bestId())return '#00e676';const i=pl?pl.cands.findIndex(c=>c.id===id):0;return ['#ffb300','#ab47bc','#29b6f6','#ef5350'][Math.max(i,0)%4]};
const clampv=(v,a,b)=>Math.max(a,Math.min(b,v));
function layerCanvas(name){if(lc.g!==g)lc={g:g};if(lc[name])return lc[name];const W=g.W,H=g.H,c=document.createElement('canvas');c.width=W*K;c.height=H*K;const x=c.getContext('2d'),im=x.createImageData(W*K,H*K);
 let cnt=null,nC=1;if(name==='corridor'&&pl){const raw=ARES.corridorHeat(g,pl.all);nC=Math.max(1,pl.all.length);cnt=new Uint16Array(W*H);for(let r=0;r<H;r++)for(let q=0;q<W;q++){let m=0;for(let a=-1;a<=1;a++)for(let b=-1;b<=1;b++){const rr=r+a,qq=q+b;if(rr>=0&&qq>=0&&rr<H&&qq<W&&raw[rr*W+qq]>m)m=raw[rr*W+qq]}cnt[r*W+q]=m}}
 let arr=name==='height'?g.Z:name==='slope'?g.slope:name==='rough'?g.rough:null,lo=0,hi=1;if(arr){lo=Infinity;hi=-Infinity;for(const v of arr){if(v<lo)lo=v;if(v>hi)hi=v}}
 let cm=null,cLo=0,cHi=1;if(name==='cost'){cm=ARES.buildM(g,TR,P.PRESETS.Balanced);cLo=Infinity;cHi=-Infinity;for(const v of cm){if(v<cLo)cLo=v;if(v>cHi)cHi=v}}
 for(let r=0;r<H;r++)for(let q=0;q<W;q++){const i=r*W+q;let col;
  if(name==='height'){const t=(g.Z[i]-lo)/(hi-lo||1),sh=0.75+0.5*g.hill[i];col=ramp(t).map(v=>Math.min(255,v*sh))}
  else if(name==='slope'||name==='rough'){const v=Math.round(255*(arr[i]-lo)/(hi-lo||1));col=[v,v,v]}
  else if(name==='hazard'){const sl=g.slope[i];col=sl>TR.max_slope_deg?[220,40,40]:sl>TR.risky_slope_deg?[240,150,40]:g.rough[i]>TR.rough_threshold_m?[240,220,70]:[60,70,80]}
  else if(name==='rad'){const t=Math.max(0,Math.min(1,(g.rad[i]-g.radMin)/g.radDen));col=t<.5?[Math.round(60+390*t),Math.round(110+290*t),Math.round(220-340*t)]:[255,Math.round(255-380*(t-.5)),30]}
  else if(name==='cost'){const t=(cm[i]-cLo)/(cHi-cLo||1);col=[Math.round(40+215*t),Math.round(200-150*t),70]}
  else if(name==='corridor'){const k=cnt?cnt[i]:0,t=k/nC;col=k?[255,255,Math.round(60+195*t),Math.round(110+145*t)]:[0,0,0,0]}
  else col=g.sand[i]?[235,225,190]:[70,60,55];
  for(let a=0;a<K;a++)for(let b=0;b<K;b++){const o=((r*K+a)*W*K+q*K+b)*4;im.data[o]=col[0];im.data[o+1]=col[1];im.data[o+2]=col[2];im.data[o+3]=col[3]===undefined?255:col[3]}}
 x.putImageData(im,0,0);lc[name]=c;return c}
function getTile(z,row,col){const key=z+'/'+row+'/'+col;let t=tileCache[key];if(t)return t;t=tileCache[key]={img:null,fail:false};const urls=P.tileUrls||[];let i=tileIdx;
 const tryLoad=()=>{if(i>=urls.length){t.fail=true;if(tilesOK!==true)tilesOK=false;redrawAll();return}
  const im=new Image();im.onload=()=>{t.img=im;tilesOK=true;tileIdx=i;redrawAll()};im.onerror=()=>{if(tilesOK===true){t.fail=true;redrawAll()}else{i++;tryLoad()}};
  im.src=urls[i].replace('{z}',z).replace('{row}',row).replace('{col}',col)};tryLoad();return t}
let _raf=0;function redrawAll(){if(_raf)return;_raf=requestAnimationFrame(()=>{_raf=0;maps.forEach(m=>m.draw())})}
function fitView(W,H){const dl=A0.lon_max-A0.lon_min,dt=A0.lat_max-A0.lat_min;view={cx:(A0.lon_min+A0.lon_max)/2,cy:(A0.lat_min+A0.lat_max)/2,s:Math.min(W/(dl*1.3),H/(dt*1.3))};view.fit=view.s}
class MapView{constructor(cv){this.cv=cv;this.o={};this.drag=null;cv._mv=this;const m=this;
 cv.addEventListener('wheel',e=>{e.preventDefault();m.zoomAt(Math.pow(1.002,-e.deltaY),e.offsetX*cv.width/cv.clientWidth,e.offsetY*cv.height/cv.clientHeight)},{passive:false});
 cv.addEventListener('pointerdown',e=>{m.drag={x:e.clientX,y:e.clientY,cx:view.cx,cy:view.cy,moved:false};try{cv.setPointerCapture(e.pointerId)}catch(_){}});
 cv.addEventListener('pointermove',e=>{const r=cv.getBoundingClientRect(),k=cv.width/r.width;if(m.drag){const dx=e.clientX-m.drag.x,dy=e.clientY-m.drag.y;if(Math.abs(dx)+Math.abs(dy)>4)m.drag.moved=true;if(m.drag.moved){view.cx=m.drag.cx-dx*k/view.s;view.cy=m.drag.cy+dy*k/view.s;redrawAll()}}
  const p=m.toGeo((e.clientX-r.left)*k,(e.clientY-r.top)*k);const c=document.getElementById(cv.id+'_pos');if(c)c.textContent=p[1].toFixed(4)+'°, '+p[0].toFixed(4)+'°'});
 cv.addEventListener('pointerup',e=>{const d=m.drag;m.drag=null;if(d&&!d.moved&&m.o.onPick){const r=cv.getBoundingClientRect(),k=cv.width/r.width,p=m.toGeo((e.clientX-r.left)*k,(e.clientY-r.top)*k);m.o.onPick(p[0],p[1])}})}
 toGeo(x,y){return[view.cx+(x-this.cv.width/2)/view.s,view.cy-(y-this.cv.height/2)/view.s]}
 toScr(lon,lat){return[this.cv.width/2+(lon-view.cx)*view.s,this.cv.height/2-(lat-view.cy)*view.s]}
 zoomAt(f,x,y){const[lo,la]=this.toGeo(x,y),ns=clampv(view.s*f,3,view.fit*12);view.s=ns;view.cx=lo-(x-this.cv.width/2)/ns;view.cy=la+(y-this.cv.height/2)/ns;redrawAll()}
 draw(){const cv=this.cv,x=cv.getContext('2d'),W=cv.width,H=cv.height,s=view.s;x.fillStyle='#2a2320';x.fillRect(0,0,W,H);x.imageSmoothingEnabled=true;
  if(tilesOK!==false){const tz=clampv(Math.ceil(Math.log2(s*180/256)),0,TMAX),td=180/Math.pow(2,tz),[l0,t0]=this.toGeo(0,0),[l1,t1]=this.toGeo(W,H);
   const c0=Math.floor((l0+180)/td),c1=Math.floor((l1+180)/td),r0=Math.floor((90-t0)/td),r1=Math.floor((90-t1)/td);
   if((c1-c0+1)*(r1-r0+1)<=80)for(let r=Math.max(0,r0);r<=Math.min(Math.pow(2,tz)-1,r1);r++)for(let c=Math.max(0,c0);c<=Math.min(Math.pow(2,tz+1)-1,c1);c++){
    const[sx,sy]=this.toScr(-180+c*td,90-r*td),sz=td*s,t=getTile(tz,r,c);
    if(t.img)x.drawImage(t.img,sx,sy,sz+1,sz+1);else{for(let up=1;up<=4&&tz-up>=0;up++){const pz=tz-up,pr=r>>up,pc=c>>up,pt=tileCache[pz+'/'+pr+'/'+pc];if(pt&&pt.img){const n=1<<up,w=pt.img.width/n,h=pt.img.height/n;x.drawImage(pt.img,(c%n)*w,(r%n)*h,w,h,sx,sy,sz+1,sz+1);break}}}}}
  const[ax,ay]=this.toScr(A0.lon_min,A0.lat_max),[bx,by]=this.toScr(A0.lon_max,A0.lat_min),act=Object.keys(ovl).filter(k=>ovl[k]);
  if(tilesOK===false&&!act.length){x.drawImage(layerCanvas('height'),ax,ay,bx-ax,by-ay)}
  act.forEach(k=>{x.globalAlpha=ovlAlpha;x.drawImage(layerCanvas(k),ax,ay,bx-ax,by-ay);x.globalAlpha=1});
  x.setLineDash([6,4]);x.strokeStyle='#fff';x.lineWidth=1.5;x.strokeRect(ax,ay,bx-ax,by-ay);x.setLineDash([]);
  ZONES.forEach(zn=>{const[zx,zy]=this.toScr(zn.lon,zn.lat),k=M_DEG(),rx=zn.radius_m/(k*Math.cos(zn.lat*Math.PI/180))*s,ry=zn.radius_m/k*s;x.beginPath();x.ellipse(zx,zy,Math.max(rx,2),Math.max(ry,2),0,0,7);x.fillStyle=zn.nogo?'rgba(150,0,0,.28)':'rgba(255,60,0,.18)';x.fill();x.setLineDash(zn.nogo?[]:[5,4]);x.strokeStyle='#ff3d00';x.lineWidth=2;x.stroke();x.setLineDash([]);x.fillStyle='#fff';x.font='bold 11px sans-serif';x.textAlign='center';x.fillText('☢ ×'+zn.mult+(zn.nogo?' no-go':''),zx,zy+4)});
  const P2=(r,c)=>{const[lo,la]=ARES.rcToLonLat(g,r,c);return this.toScr(lo,la)},pt=([lo,la],col,t)=>{const[px,py]=this.toScr(lo,la);x.fillStyle=col;x.beginPath();x.arc(px,py,9,0,7);x.fill();x.strokeStyle='#fff';x.lineWidth=2;x.stroke();x.fillStyle='#fff';x.font='bold 11px sans-serif';x.textAlign='center';x.fillText(t,px,py+4)};
  const o=this.o;if(o.paths&&pl){const bid=bestId();(o.list||fms).forEach(m=>{const cd=pl.cands.find(c=>c.id===m.id);if(!cd)return;const on=m.id===sel,isb=m.id===bid;x.beginPath();cd.path.forEach(([r,c],i)=>{const[px,py]=P2(r,c);i?x.lineTo(px,py):x.moveTo(px,py)});if(on){x.strokeStyle='#fff';x.lineWidth=7;x.stroke()}if(isb){x.strokeStyle='#fff';x.lineWidth=9;x.stroke()}x.strokeStyle=pcol(m.id);x.lineWidth=isb?5:(on?4:2.5);x.setLineDash(m.hard_fail?[7,5]:[]);x.stroke();x.setLineDash([])});
   const m=fms.find(m=>m.id===sel);if(m&&o.pnr&&m.pnr_idx!=null&&!m.pnr_none){const cd=pl.cands.find(c=>c.id===m.id),p=cd.path[Math.min(m.pnr_idx,cd.path.length-1)];if(p)pt(p,'#a020f0','N')}}
  if(o.paths&&pl&&pl.all&&pl.byLayer)Object.keys(ovl).filter(k=>ovl[k]&&pl.byLayer[k]).forEach(k=>{const cd=pl.all.find(c=>c.id===pl.byLayer[k]);if(!cd)return;x.beginPath();cd.path.forEach(([r,c],i)=>{const[px,py]=P2(r,c);i?x.lineTo(px,py):x.moveTo(px,py)});x.strokeStyle='#fff';x.lineWidth=8;x.stroke();x.strokeStyle='#00e5ff';x.lineWidth=4;x.setLineDash([8,4]);x.stroke();x.setLineDash([]);const mid=cd.path[cd.path.length>>1],[mx_,my_]=P2(mid[0],mid[1]);x.fillStyle='#00e5ff';x.font='bold 11px sans-serif';x.textAlign='center';x.fillText('best for '+k,mx_,my_-8)});
  if(S)pt(S,'#1976d2','S');if(E)pt(E,'#c62828','E');
  const km=(100/s)*(A0.lat_max-A0.lat_min>0?1:1);const mpx=(M_DEG()/s);x.fillStyle='rgba(0,0,0,.55)';x.fillRect(W-130,H-26,126,22);x.fillStyle='#fff';x.font='11px sans-serif';x.textAlign='left';const nice=[10,20,50,100,200,500,1000,2000,5000,10000,20000,50000,100000].find(v=>v/mpx>=50)||100000;x.fillRect(W-124,H-12,nice/mpx,3);x.fillText(nice>=1000?nice/1000+' km':nice+' m',W-124,H-16);
  const st=document.getElementById(cv.id+'_st');if(st)st.textContent=tilesOK===false?'⚠ NASA imagery could not load (no internet or tile address changed): showing the height map instead.':(tilesOK===null?'Loading NASA imagery…':'')}}
const M_DEG=()=>3396190*Math.PI/180;
function mapHTML(id,h){return `<div class="mapw"><canvas id="${id}" data-h="${h||520}"></canvas>
<div class="lyr"><button class="btn" id="${id}_lb">☰ Layers</button><div class="lpn" id="${id}_lp" style="display:none">${LAYERS.map(l=>`<label title="${l[2]}"><input type="checkbox" data-l="${l[0]}" ${ovl[l[0]]?'checked':''}> ${l[1]} <span class="mut">(${l[2]})</span></label>`).join('')}<label>Opacity <input type="range" min="0.1" max="1" step="0.05" value="${ovlAlpha}" data-o="1"></label></div></div>
<div class="zb"><button class="btn" data-z="1">＋</button><button class="btn" data-z="-1">－</button><button class="btn" data-z="0" title="Show the whole area">⌖</button></div>
<div class="cap"><span id="${id}_pos"></span></div></div><div class="mut" id="${id}_st"></div><div class="mut" style="font-size:11px">Imagery: NASA/JPL Mars Trek (Viking colour mosaic, 232 m per pixel: it gets blurry when zoomed in). Heights: ${esc(P.label)}. Scroll or use ＋ － to zoom, drag to move.</div>`}
function mapInit(id,opts){const cv=$(id);if(!cv)return null;const w=cv.parentElement.clientWidth||640;cv.width=Math.min(900,Math.max(320,w));cv.height=Math.round(+cv.dataset.h*Math.min(1,cv.width/640)||360);
 if(!view)fitView(cv.width,cv.height);const mv=cv._mv||new MapView(cv);mv.o=opts||{};maps.push(mv);
 const lb=$(id+'_lb'),lp=$(id+'_lp');lb.onclick=()=>{lp.style.display=lp.style.display==='none'?'block':'none'};
 lp.onchange=e=>{const t=e.target;if(t.dataset.l)ovl[t.dataset.l]=t.checked;redrawAll()};lp.oninput=e=>{if(e.target.dataset.o){ovlAlpha=+e.target.value;redrawAll()}};
 cv.parentElement.querySelector('.zb').onclick=e=>{const b=e.target.closest('button');if(!b)return;const z=+b.dataset.z;if(z===0){fitView(cv.width,cv.height);redrawAll()}else mv.zoomAt(z>0?1.5:1/1.5,cv.width/2,cv.height/2)};
 mv.draw();return mv}
function defaultRoute(){const H=g.H,W=g.W,cols=Math.round(1250/g.px),r1=Math.round(H*.55),c1=Math.round(W*.05),r2=Math.round(H*.42),c2=Math.min(W-2,c1+cols);S=ARES.rcToLonLat(g,r1,c1);E=ARES.rcToLonLat(g,r2,c2)}
function pickLL(lon,lat){if(zoneMode){addZone(lon,lat);return}const p=[lon,lat];if(!S||(S&&E)){S=p;E=null}else E=p;render()}
function zonesChanged(){g=ARES.makeGrid(D,TR,ZONES);lc={g:null};pl=null;tms=[];fms=[];render()}
function addZone(lon,lat){const r=parseFloat($('zr').value)||300,m=parseFloat($('zm').value)||3;ZONES.push({name:'Zone '+(ZONES.length+1),lon,lat,radius_m:Math.max(20,r),mult:Math.max(1,m),nogo:$('zn').checked});zoneMode=false;zonesChanged()}
function zoneList(){const el=$('zlist');if(!el)return;const zb=$('zpick');if(zb)zb.textContent=zoneMode?'Now click the map…':'Click map to place a zone';
 el.innerHTML=ZONES.length?ZONES.map((z,i)=>`<div>☢ ${esc(z.name)} · ${z.radius_m} m · ×${z.mult}${z.nogo?' · NO-GO':''} · ${z.lat.toFixed(4)}, ${z.lon.toFixed(4)} <button class="btn" data-zd="${i}">remove</button></div>`).join(''):'<span class="mut">No zones yet. Without a zone the radiation layer only shows terrain shielding, which differs very little over a few km.</span>';
 el.onclick=e=>{const b=e.target.closest('button');if(!b)return;ZONES.splice(+b.dataset.zd,1);zonesChanged()}}
const rcOf=p=>ARES.lonLatToRc(g,p[0],p[1]);
const mDist=(a,b)=>{const k=M_DEG(),la=(a[1]+b[1])/2*Math.PI/180;return Math.hypot((a[0]-b[0])*Math.cos(la)*k,(a[1]-b[1])*k)};
/* ---------- planning ---------- */
const inArea=p=>{const a=A0,mx=(a.lon_max-a.lon_min)*.03,my=(a.lat_max-a.lat_min)*.03;return p[0]>a.lon_min+mx&&p[0]<a.lon_max-mx&&p[1]>a.lat_min+my&&p[1]<a.lat_max-my};
let _lt=null;async function loadArea(){const t0=Date.now();try{await loadArea0()}finally{_lt=((Date.now()-t0)/1000)}}
async function loadArea0(){if(!(window.google&&google.colab&&google.colab.kernel))throw new Error('These points are outside the loaded area. Loading a new area works in Google Colab only: there, press Find paths again. Elsewhere, set CENTER in cell 4 and re-run cells 4-5.');
 const r=await google.colab.kernel.invokeFunction('ares_load',[S[0],S[1],E[0],E[1]],{}),pj=r&&r.data&&r.data['application/json'];if(!pj)throw new Error('The Python side returned nothing. Run cell 5 again, then retry.');
 const keep={tileUrls:P.tileUrls,tileMaxZoom:P.tileMaxZoom};Object.assign(P,pj,keep);window._pyT=pj.timing||null;D=ARES.decodeData(P);A0=P.area;g=ARES.makeGrid(D,TR,ZONES);lc={g:null};banner()}
let _pt=0;async function plan(){_lt=null;const tp=Date.now();if(!S||!E){alert('Pick START and END first.');return false}
 if(!(inArea(S)&&inArea(E))){const m=$('runmsg');if(m)m.textContent='Loading real height data for this part of Mars…';await loadArea()}
 const dm=mDist(S,E);if(dm/TR.leg_length_m>60){TR.leg_length_m=Math.ceil(dm/40/50)*50;const inp=document.querySelector('#trset [data-k=leg_length_m]');if(inp)inp.value=TR.leg_length_m}
 const Sr=rcOf(S),Er=rcOf(E);pl=ARES.planFeatures(g,TR,Sr,Er,P.PRESETS,tw);if(lc)lc.corridor=null;
 tms=pl.cands.map(c=>ARES.measureTerrain(g,TR,c));fms=ARES.runStage2(g,TR,A,tms);infl=null;_pt=(Date.now()-tp)/1000;sel=(ARES.scorePaths(tms,ARES.TERRAIN_KEYS,tw)[0]||{}).id;return true}
function recomputeFull(){if(tms.length){fms=ARES.runStage2(g,TR,A,tms);infl=null}}
/* ---------- pages ---------- */
function ll(p){return p[1].toFixed(4)+', '+p[0].toFixed(4)}
function p1(){const el=$('p1');if(!el.dataset.b){el.dataset.b=1;if(!S)defaultRoute();el.innerHTML=`<div class="card"><b>Click anywhere on Mars</b>: first click sets START (blue S), second sets END (red E), a third click starts over. Zoom out to see the whole planet. The dashed box is the area whose height data is loaded now; points outside it load a new area (Colab). A default walk of about 30 minutes is already set: press <b>Find paths</b>, or move the points.</div>
<div class="row"><div>${mapHTML('map1',520)}</div>
<div><div class="card"><div id="pick"></div>
<label class="s">START lat<input id="slat" type="number" step="0.0001"></label><label class="s">START lon<input id="slon" type="number" step="0.0001"></label>
<label class="s">END lat<input id="elat" type="number" step="0.0001"></label><label class="s">END lon<input id="elon" type="number" step="0.0001"></label>
<button class="btn" id="setll">Use typed values</button> <button class="btn" id="demoPick">Default 30-min route</button></div>
<div class="card"><b>Terrain rules</b> <span class="mut">(stage 1, no assumptions)</span><div id="trset"></div></div>
<div class="card"><b>☢ Radiation zones</b> <span class="mut">(hypothetical circles: the planner treats them as extra radiation; real Mars radiation does not come in circles)</span>
<label class="s">Radius (m)<input id="zr" type="number" value="300"></label><label class="s">Strength (× open-ground dose)<input id="zm" type="number" step="0.5" value="3"></label><label class="s">No-go (hard block)<input id="zn" type="checkbox"></label>
<button class="btn" id="zpick">Click map to place a zone</button> <button class="btn" id="zmid">Demo zone on the straight line</button><div id="zlist" style="margin-top:6px"></div></div>
<button class="btn pri" id="runb">Find paths</button> <span id="runmsg" class="mut"></span></div></div>`;
 $('setll').onclick=()=>{const v=id=>parseFloat($(id).value);if([v('slat'),v('slon'),v('elat'),v('elon')].some(isNaN))return;S=[v('slon'),v('slat')];E=[v('elon'),v('elat')];render()};
 $('demoPick').onclick=()=>{defaultRoute();render()};
 $('zpick').onclick=()=>{zoneMode=!zoneMode;zoneList()};
 $('zmid').onclick=()=>{if(!S||!E){alert('Pick START and END first.');return}const d=mDist(S,E);ZONES.push({name:'Zone '+(ZONES.length+1),lon:(S[0]+E[0])/2,lat:(S[1]+E[1])/2,radius_m:Math.max(60,Math.round(d*0.15/10)*10),mult:parseFloat($('zm').value)||3,nogo:$('zn').checked});zonesChanged()};
 $('runb').onclick=async()=>{$('runmsg').textContent='Working…';try{const t=Date.now();if(await plan()){$('runmsg').textContent=pl.cands.length+' routes in '+_pt.toFixed(1)+' s'+(_lt!=null?' (new map data: '+_lt.toFixed(1)+' s'+(window._pyT?' = download '+window._pyT.download+' s + layers '+window._pyT.layers+' s + transfer ≈ '+Math.max(0,_lt-window._pyT.download-window._pyT.layers-window._pyT.pack).toFixed(1)+' s':'')+')':' (map data already loaded: no download)');go(2)}else $('runmsg').textContent=''}catch(e){$('runmsg').textContent='⚠ '+e.message}};
 $('trset').innerHTML=settingsHTML(P.TR_SPEC,TR,'tr');bindSettings('trset',P.TR_SPEC,TR,()=>{g=ARES.makeGrid(D,TR,ZONES);lc={g:null};pl=null;tms=[];fms=[];render()})}
 mapInit('map1',{paths:false,onPick:pickLL});zoneList();
 let info='';if(S&&E){const d=mDist(S,E),mn=d/(A.flat_speed_kmh*1000/60);info=`<br>Straight line: <b>${(d/1000).toFixed(2)} km</b> ≈ <b>${mn<120?Math.round(mn)+' min':(mn/60).toFixed(1)+' h'}</b> on flat ground at ${A.flat_speed_kmh} km/h (real paths are longer)`+(d>8000?'<br><span class="wn">⚠ Long route: a suited walk this long will probably break the oxygen/battery rules. The planner still ranks it; check pages 4 and 6.</span>':'')+(inArea(S)&&inArea(E)?'':'<br><span class="mut">Outside the loaded area: new height data is fetched when you press Find paths (Colab).</span>')}
 $('pick').innerHTML=`START: <b>${S?ll(S):'not set'}</b><br>END: <b>${E?ll(E):'not set'}</b>${info}`}
function settingsHTML(spec,obj,pre){let h='',grp='';spec.forEach(s=>{if(s.group!==grp){grp=s.group;h+=`<h2>${esc(grp)}</h2>`}
 const inp=s.kind==='choice'?`<select data-k="${s.key}">${(s.options||[]).map(o=>`<option ${o==obj[s.key]?'selected':''}>${esc(o)}</option>`).join('')}</select>`:`<input data-k="${s.key}" type="number" step="any" min="${s.lo??''}" max="${s.hi??''}" value="${obj[s.key]}">`;
 h+=`<label class="s" title="${esc(s.note)}"><span>${esc(s.label)} <span class="mut">(${esc(s.unit)})</span></span>${inp}</label>`});return h}
function bindSettings(id,spec,obj,cb){$(id).onchange=e=>{const k=e.target.dataset.k;if(!k)return;const s=spec.find(x=>x.key===k);obj[k]=s.kind==='choice'?e.target.value:parseFloat(e.target.value);cb()}}
function tbl(rows,cols,clickable){return `<div class="sc"><table><tr>${cols.map(c=>`<th>${c[0]}</th>`).join('')}</tr>${rows.map(r=>`<tr class="${clickable?'cl':''} ${r.id===sel?'sel':''}" data-id="${r.id}">${cols.map(c=>`<td>${c[1](r)}</td>`).join('')}</tr>`).join('')}</table></div>`}
const need=()=>!pl?'<div class="card">Pick START and END on page 1 and press <b>Find paths</b>.</div>':'';
function selRows(el){el.querySelectorAll('tr.cl').forEach(r=>r.onclick=()=>{sel=r.dataset.id;render()})}
const LAYER_BEST=[['','— choose a layer —',null],['cost','Why: total cost (the best route)',null],['height','Height: least climbing','climb_m'],['slope','Slope: flattest ground','mean_slope_deg'],['rough','Roughness: smoothest','pct_rough'],['hazard','Hazard: most clearance','risk'],['sand','Sand: least sand','pct_sand'],['rad','Radiation: lowest exposure','rad_idx']];
function p2(){const el=$('p2');if(!pl){el.innerHTML=need();return}
 const rk=ARES.scorePaths(tms,ARES.TERRAIN_KEYS,tw),order=rk.map(r=>r.id),bid=order[0],rows=order.map(id=>tms.find(m=>m.id===id));
 el.innerHTML=`<div class="card">Terrain only. No crew, oxygen or battery assumptions are used on this page. Reference time is Tobler's rule for an unburdened Earth hiker, not a suited astronaut.</div>
 <div class="card" id="spread"></div>
 <div class="card"><label>Layer view (heat map + the route built only for that layer): <select id="lbs">${LAYER_BEST.map(x=>`<option value="${x[0]}" ${x[0]===layerPick?'selected':''}>${x[1]}</option>`).join('')}</select></label><div id="lbmsg" style="margin-top:6px"></div></div>
 <div class="row"><div>${mapHTML("map2",460)}</div><div id="t2"></div></div><h2>Leg table for ${esc(sel||'')}</h2><div id="legs2"></div>`;
 mapInit('map2',{paths:true,list:rows});
 $('spread').innerHTML=`<b>★ ${esc(bid)} is the best route</b> (thick green line, lowest terrain score). It was picked from ${pl.generated} routes, each built for ONE feature (shortest, least climb, flattest, smoothest, most clearance, least sand, lowest radiation, balanced). The other ${rows.length-1} shown are the most useful alternatives. <span class="mut">Dashed = fails a hard rule.</span>${pl.notes.length?'<div class="wn">'+pl.notes.map(esc).join('<br>')+'</div>':''}`;
 const lm=$('lbmsg');
 if(!layerPick)lm.innerHTML='<span class="mut">Pick a layer: the map shows its heat map and the route built only for that layer (dotted cyan) next to the best route (green). The ☰ Layers button on the map does the same.</span>';
 else if(layerPick==='cost')lm.innerHTML=`Heat map = the price of walking over each square, all features mixed (green = cheap, red = expensive). The best route <b>${esc(bid)}</b> bends toward the cheap squares.`;
 else{const spec=LAYER_BEST.find(x=>x[0]===layerPick),k=spec[2],bm=tms.find(m=>m.id===bid),aid=pl.byLayer[layerPick],am=pl.allM.find(m=>m.id===aid);
  if(!am)lm.innerHTML='<span class="wn">No route for this layer'+(layerPick==='sand'&&!g.hasSand?': no sand mask is loaded.':'.')+'</span>';
  else{const same=ARES.pathSep(pl.all.find(c=>c.id===aid).path,pl.cands.find(c=>c.id===bid).path)<1.5,p=(am.dist_m/bm.dist_m-1)*100;
   lm.innerHTML=same?`The best route <b>${esc(bid)}</b> already follows the best route for this layer (${esc(CR[k].label)} ${fmt(k,bm[k])}).`:`Route built only for <b>${esc(spec[1].split(':')[0])}</b> (dotted cyan): ${esc(CR[k].label)} <b>${fmt(k,am[k])}</b> against <b>${fmt(k,bm[k])}</b> on the best route ${esc(bid)}, but it is ${Math.abs(p)<1?'the same length':Math.abs(p).toFixed(0)+'% '+(p>0?'longer':'shorter')}. The best route trades this layer against the others.`}}
 $('lbs').onchange=e=>{layerPick=e.target.value;Object.keys(ovl).forEach(k=>ovl[k]=false);if(layerPick)ovl[layerPick]=true;render()};
 $('t2').innerHTML=tbl(rows,[['#',r=>order.indexOf(r.id)+1],['Path',r=>`<span style="display:inline-block;width:12px;height:12px;border-radius:3px;background:${pcol(r.id)};vertical-align:-2px"></span> `+(r.id===bid?'★ ':'')+esc(r.id)+' '+esc(r.label)],['Why this route',r=>`<span style="white-space:normal;display:inline-block;min-width:260px">${esc((pl.why&&pl.why[r.id])||'')}</span>`],['Dist',r=>fmt('dist_m',r.dist_m)],['Ref time',r=>ARES.fmtHm(r.ref_time_min)],['Climb',r=>fmt('climb_m',r.climb_m)],['Max slope',r=>fmt('max_slope_deg',r.max_slope_deg)],['Steep',r=>fmt('pct_steep',r.pct_steep)],['Sand',r=>fmt('pct_sand',r.pct_sand)],['Rad',r=>fmt('rad_idx',r.rad_idx)],['Problems',r=>r.problems.length?r.problems.map(p=>`<span class="${p[0]==='HIGH'?'bad':'wn'}">${p[0]}</span> ${esc(p[1].slice(0,70))}…`).join('<br>'):'<span class="ok">none</span>']],true);
 selRows($('t2'));const m=tms.find(x=>x.id===sel);if(m)$('legs2').innerHTML=`<div class="mut">${esc(m.head)}</div>`+tbl(m.legs.map((l,i)=>({...l,id:i})),Object.keys(m.legs[0]).map(k=>[esc(k),r=>typeof r[k]==='number'?r[k].toFixed(1):esc(r[k])]),false)}
function p3(){const el=$('p3');if(!el.dataset.b){el.dataset.b=1;el.innerHTML=`<div class="card">Assumptions are guesses about the crew, suit and rover. They are applied to the <b>same paths</b> without searching again. Changing a value updates every other page. Defaults decide a lot: check the battery size and power draw with your team.</div><div id="aset"></div><button class="btn" id="areset">Reset to defaults</button>`;
  $('aset').innerHTML=settingsHTML(P.A_SPEC,A,'a');bindSettings('aset',P.A_SPEC,A,()=>{recomputeFull();$('amsg').textContent=''});
  $('areset').onclick=()=>{P.A_SPEC.forEach(s=>A[s.key]=s.default);$('aset').innerHTML=settingsHTML(P.A_SPEC,A,'a');recomputeFull()};el.insertAdjacentHTML('beforeend','<div id="amsg"></div>')}
 }
function p4(){const el=$('p4');if(!pl){el.innerHTML=need();return}
 const rt=ARES.scorePaths(tms,ARES.TERRAIN_KEYS,tw),rf=ARES.scorePaths(fms,ARES.FULL_KEYS,fw),rkT={},rkF={};rt.forEach(r=>rkT[r.id]=r.rank);rf.forEach(r=>rkF[r.id]=r.rank);
 el.innerHTML=`<h2>Before and after the assumptions</h2><div id="b4"></div><h2>Which assumptions matter most?</h2><div class="mut">Each assumption is moved ±25%, one at a time. Takes a few seconds.</div><button class="btn" id="inflb">${infl?'Recompute':'Run sensitivity test'}</button><div id="inf"></div><h2>Summary</h2><div id="sum"></div>`;
 $('b4').innerHTML=tbl(fms,[['Path',r=>esc(r.id)+' '+esc(r.label)],['Ref time',r=>ARES.fmtHm(r.ref_time_min)],['Suited time',r=>ARES.fmtHm(r.time_min)],['Oxygen used',r=>fmt('o2_used_pct',r.o2_used_pct)],['Battery used',r=>fmt('batt_used_pct',r.batt_used_pct)],['Rank (terrain)',r=>rkT[r.id]],['Rank (full)',r=>rkF[r.id]],['Result',r=>r.hard_fail?`<span class="bad">FAIL</span> ${esc((r.fails||[]).join('; '))}`:'<span class="ok">pass</span>']],false);
 const draw=()=>{if(!infl)return;$('inf').innerHTML=tbl(infl.slice(0,12).map((r,i)=>({...r,id:i})),[['Assumption',r=>esc(r.Assumption)],['Now',r=>r.Now+' '+esc(r.Unit)],['Affects',r=>esc(r.Affects)],['Paths flipping pass/fail',r=>r.Flips],['Winner changes',r=>r.WinnerChanges],['Impact',r=>`<b class="${r.Impact==='HIGH'?'bad':r.Impact==='MEDIUM'?'wn':'mut'}">${r.Impact}</b>`]],false);
  $('sum').innerHTML=ARES.plainSummary(tms,fms,infl,tw,fw).map(l=>`<p>${esc(l)}</p>`).join('')};draw();
 $('inflb').onclick=()=>{$('inflb').textContent='Working…';setTimeout(()=>{infl=ARES.assumptionInfluence(g,TR,A,P.A_SPEC,tms,fw);draw();$('inflb').textContent='Recompute'},30)}}
function p5(){const el=$('p5');if(!pl){el.innerHTML=need();return}
 const T=stage==='terrain',keys=T?ARES.TERRAIN_KEYS:ARES.FULL_KEYS,W=T?tw:fw,pr=T?ARES.TERRAIN_PRIORITIES:ARES.FULL_PRIORITIES,ms=T?tms:fms;
 const rk=ARES.scorePaths(ms,keys,W),ex=ARES.explainRanking(rk,keys,W);
 el.innerHTML=`<div class="card">Stage: <select id="stg"><option value="terrain" ${T?'selected':''}>Terrain only</option><option value="full" ${!T?'selected':''}>With assumptions</option></select> Priority: <select id="prf">${Object.keys(pr).map(k=>`<option ${k===prof[T?'t':'f']?'selected':''}>${k}</option>`).join('')}</select>
 <div class="mut" style="margin-top:6px">${esc(ARES.formulaText(keys,W))}</div></div>
 <div class="row"><div class="card" id="wsl"></div><div id="rk"></div></div>`;
 $('wsl').innerHTML=keys.map(k=>`<label class="s"><span>${CR[k].label} <b>${((W[k]||0)*100).toFixed(0)}</b></span><input type="range" min="0" max="1" step="0.01" data-k="${k}" value="${W[k]||0}"></label>`).join('');
 $('rk').innerHTML=tbl(rk.map(r=>{const e=ex.strengths.find(s=>s.id===r.id)||{};const m=ms.find(x=>x.id===r.id);return {id:r.id,label:r.label,rank:r.rank,score:r.score,fail:m.hard_fail,st:e.Strengths,wk:e.Weaknesses}}),[['#',r=>r.rank],['Path',r=>esc(r.id)+' '+esc(r.label)],['Score',r=>r.score.toFixed(3)],['Hard rules',r=>r.fail?'<span class="bad">not recommended</span>':'<span class="ok">ok</span>'],['Strengths',r=>esc(r.st||'')],['Weaknesses',r=>esc(r.wk||'')]],true);selRows($('rk'));
 $('stg').onchange=e=>{stage=e.target.value;render()};
 $('prf').onchange=e=>{const k=e.target.value;prof[T?'t':'f']=k;Object.keys(W).forEach(x=>delete W[x]);Object.assign(W,pr[k]);render()};
 $('wsl').oninput=e=>{const k=e.target.dataset.k;if(!k)return;W[k]=parseFloat(e.target.value);const r2=ARES.scorePaths(ms,keys,W);clearTimeout(window._t);window._t=setTimeout(render,120)}}
function p6(){const el=$('p6');if(!pl){el.innerHTML=need();return}
 const m=fms.find(x=>x.id===sel)||fms[0];sel=m.id;
 el.innerHTML=`<div class="card">Path: <select id="ps">${fms.map(x=>`<option value="${x.id}" ${x.id===sel?'selected':''}>${esc(x.id)} ${esc(x.label)}${x.hard_fail?' (fails)':''}</option>`).join('')}</select> ${m.hard_fail?'<b class="bad">NOT RECOMMENDED</b>':'<b class="ok">passes hard rules</b>'}<div class="mut">${esc(m.head)}</div></div>
 <div class="row"><div>${mapHTML("map6",460)}</div><div><div class="card"><table>
 <tr><td>Suited walk time</td><td>${ARES.fmtHm(m.time_min)}</td></tr><tr><td>Oxygen used / left</td><td>${m.o2_used_pct.toFixed(0)}% / ${m.o2_left_pct.toFixed(0)}%</td></tr>
 <tr><td>Battery used / left</td><td>${m.batt_used_pct.toFixed(0)}% / ${m.batt_left_pct.toFixed(0)}%</td></tr><tr><td>Radiation</td><td>${m.rad_msv.toFixed(3)} mSv</td></tr>
 <tr><td>Return trip</td><td>${ARES.fmtHm(m.rt_time_min)} · O₂ ${m.rt_o2_pct.toFixed(0)}% · battery ${m.rt_batt_pct.toFixed(0)}%</td></tr>
 <tr><td>Point of no return</td><td>${m.pnr_none?'none: full return is possible':m.pnr_km.toFixed(1)+' km (purple N on map)'}</td></tr><tr><td>Full return OK</td><td>${m.full_return_ok?'<span class="ok">yes</span>':'<span class="bad">no</span>'}</td></tr></table></div>
 ${(m.fails||[]).map(f=>`<div class="bad">✗ ${esc(f)}</div>`).join('')}${(m.warns||[]).map(w=>`<div class="wn">⚠ ${esc(w[0])}</div>`).join('')}</div></div>
 <h2>Height profile</h2><canvas id="prof" width="900" height="180"></canvas><h2>Legs</h2><div id="lg"></div>`;
 $('ps').onchange=e=>{sel=e.target.value;render()};mapInit('map6',{paths:true,pnr:true,list:fms});
 const c=$('prof'),x=c.getContext('2d'),z=m.arr.z,d=m.arr.cumd,zl=Math.min(...z),zh=Math.max(...z),dm=d[d.length-1]||1;x.fillStyle=getComputedStyle(document.body).color;x.strokeStyle='#ff7a45';x.lineWidth=2;x.beginPath();z.forEach((v,i)=>{const px=10+d[i]/dm*880,py=165-(v-zl)/(zh-zl||1)*140;i?x.lineTo(px,py):x.moveTo(px,py)});x.stroke();x.font='11px sans-serif';x.fillText(zh.toFixed(0)+' m',12,14);x.fillText(zl.toFixed(0)+' m',12,178);x.fillText((dm/1000).toFixed(1)+' km',850,178);
 if(m.legs_full)$('lg').innerHTML=tbl(m.legs_full.map((l,i)=>({...l,id:i})),Object.keys(m.legs_full[0]).map(k=>[esc(k),r=>typeof r[k]==='number'?r[k].toFixed(1):esc(r[k])]),false)}
function p7(){const el=$('p7');if(!pl){el.innerHTML=need();return}
 const keys=['id','label','origin','fam','hard_fail','dist_m','ref_time_min','climb_m','max_slope_deg','pct_steep','pct_sand','risk','time_min','o2_used_pct','batt_used_pct','rad_msv','rad_idx','pnr_km'];
 const csv=[keys.join(',')].concat(fms.map(m=>keys.map(k=>typeof m[k]==='number'?+m[k].toFixed(3):'"'+String(m[k]).replace(/"/g,'""')+'"').join(','))).join('\n');
 const audit={terrain_data:P.label,sand:P.sandLabel,is_demo:P.isDemo,start:S&&ll(S),end:E&&ll(E),terrain_rules:TR,assumptions:A,weights:{terrain:tw,full:fw},radiation_zones:ZONES};
 el.innerHTML=`<div class="card">${P.isDemo?'<b class="wn">DEMO — not Mars.</b> ':''}Results table (CSV). Select all and copy.</div><textarea id="csv" readonly>${esc(csv)}</textarea><button class="btn" onclick="navigator.clipboard&&navigator.clipboard.writeText($('csv').value)">Copy CSV</button>
 <h2>Audit: every setting behind these numbers</h2><textarea id="aud" readonly>${esc(JSON.stringify(audit,null,1))}</textarea><button class="btn" onclick="navigator.clipboard&&navigator.clipboard.writeText($('aud').value)">Copy audit</button>`}
function render(){maps=[];[null,p1,p2,p3,p4,p5,p6,p7][page]()}
go(1);
</script></body></html>
'''


In [ ]:
# Cell 4 · REAL MARS DATA — choose the area, then load the height map
# This is only the FIRST area shown when the app opens. In the app you can click ANYWHERE on Mars: new height data is fetched for the points you pick (Colab).
SITES = {   # (lat, lon) centre of the small walking area
    "Gale crater: Curiosity landing site (Bradbury)": (-4.589, 137.441),
    "Jezero crater: Perseverance landing site":       (18.445, 77.451),
    "Custom (type your own lat/lon below)":           (-4.589, 137.441),
}
SITE = "Gale crater: Curiosity landing site (Bradbury)"
CENTER = SITES[SITE]                      # custom: CENTER = (lat, lon)
AREA_KM = 3.0      # width/height of the area in km. 3 km fits a ~30 minute suited walk. Make it bigger for longer walks.
_dlat = (AREA_KM / 2) * 1000 / M_PER_DEG
_dlon = _dlat / math.cos(math.radians(CENTER[0]))
AREA = dict(lat_min=CENTER[0] - _dlat, lat_max=CENTER[0] + _dlat, lon_min=CENTER[1] - _dlon, lon_max=CENTER[1] + _dlon)

# Real NASA picture (what you see by default, like a map app): Mars Trek, Viking colour mosaic. Several addresses are tried in order.
TILE_URLS = [
    "https://trek.nasa.gov/tiles/Mars/EQ/Mars_Viking_MDIM21_ClrMosaic_global_232m/1.0.0/default/default028mm/{z}/{row}/{col}.jpg",
    "https://trek.nasa.gov/tiles/Mars/EQ/Viking_MDIM21_ClrMosaic_global_232m/1.0.0/default/default028mm/{z}/{row}/{col}.jpg",
]
TILE_MAX_ZOOM = 7

# Real height map: USGS MOLA+HRSC blended DEM, 200 m per pixel, read straight from the internet (only your box is downloaded).
DEM_URL  = "/vsicurl/https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif"
DEM_FILE = ""      # OR the path of a GeoTIFF you exported from Mars Trek (used first if filled in)
SAND_FILE = ""     # optional sand / rock-unit mask picture or GeoTIFF for the same box

_ENV = None
def _gdal_env():
    global _ENV
    if _ENV is None and HAVE_RASTERIO:
        import rasterio
        _ENV = rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif", GDAL_HTTP_TIMEOUT="60", GDAL_HTTP_MULTIPLEX="YES",
                            VSI_CACHE="TRUE", VSI_CACHE_SIZE="100000000", CPL_VSIL_CURL_CACHE_SIZE="200000000", CPL_VSIL_CURL_CHUNK_SIZE="1048576", GDAL_CACHEMAX="256")
        _ENV.__enter__()      # stays open for the whole session

def get_terrain():
    src = DEM_FILE if DEM_FILE and os.path.exists(DEM_FILE) else DEM_URL
    if HAVE_RASTERIO:
        try:
            _gdal_env()
            T = load_terrain(DATA_SOURCES[1], AREA, TR["max_grid_cells"], dem_file=src, sand_file=SAND_FILE) if os.path.exists(src) else _load_remote(src)
            if src == DEM_URL: T.label = "USGS MOLA+HRSC blended DEM 200 m (real Mars)"
            return T
        except Exception as ex:
            print("Could not load the real height map:", type(ex).__name__, str(ex)[:300])
    print("-> Using DEMO terrain (made up, NOT Mars). Fix the message above (or set DEM_FILE) and run this cell again.")
    return make_demo_terrain(AREA, TR["max_grid_cells"])

def _load_remote(url):
    w_m, h_m = area_size_m(AREA)
    px = max(w_m, h_m) / TR["max_grid_cells"]
    shape = (max(30, int(round(h_m / px))), max(30, int(round(w_m / px))))
    Z, native = _read_dem_file(url, AREA, shape)
    notes = []
    if native and native > px * 1.5:
        notes.append(f"DEM native resolution about {native:.0f} m is coarser than the grid ({px:.0f} m): slopes are UNDERESTIMATED.")
    sand, sl = None, "no sand data (sand = 0)"
    if SAND_FILE:
        sand = _read_mask_file(SAND_FILE, shape); sl = "sand mask file"
    return Terrain(Z, px, AREA, "real DEM", False, sand=sand, sand_label=sl, notes=notes, native_res_m=native)

T = get_terrain()
L = build_layers(T, TR)
print(f"{T.label}\n  height range {np.nanmin(T.Z):.0f} to {np.nanmax(T.Z):.0f} m, grid {T.H} x {T.W}, cell {T.px:.0f} m, demo={T.is_demo}")


In [ ]:
# Cell 5 · START THE APP (v5): click START and END ANYWHERE on the Mars map, press "Find paths", then open pages 2-7.
import re
_EXTRA_RULES = [
    dict(key="max_extra_pct", label="Longest route vs the shortest found", default=100, unit="% longer", group="Route search", tag="THRESHOLD", note="Any route longer than this compared with the SHORTEST route found is dropped, however different it is.", kind="num", lo=0, hi=200, options=None, info=False),
    dict(key="main_extra_pct", label="Main route may be this much longer than the shortest", default=60, unit="% longer", group="Route search", tag="THRESHOLD", note="The main (best) route is chosen only among routes within this limit of the shortest one, so the main route is never a long detour.", kind="num", lo=0, hi=100, options=None, info=False),
    dict(key="adaptive_contrast", label="Scale slope/roughness cost to this area", default=1, unit="1 = on", group="Route search", tag="THRESHOLD", note="On: the route styles react to the slopes that really exist here, so they differ even on gentle ground. Hard limits are not changed.", kind="num", lo=0, hi=1, options=None, info=False),
    dict(key="climb_penalty", label="Climb cost multiplier", default=6, unit="x", group="Avoid mountains", tag="THRESHOLD", note="Every metre of climbing costs this many times more. Higher = routes go around hills instead of over them.", kind="num", lo=0, hi=60, options=None, info=False),
    dict(key="slope_penalty", label="Slope cost multiplier", default=3, unit="x", group="Avoid mountains", tag="THRESHOLD", note="Makes up and down slopes cost more in the route price. Higher = flatter routes, even if longer.", kind="num", lo=0, hi=20, options=None, info=False),
    dict(key="height_penalty", label="Avoid high ground", default=5, unit="x", group="Avoid mountains", tag="THRESHOLD", note="Extra price for ground higher than the START/END, up to this many times the normal price at the highest point. Pushes routes around mountains and plateaus.", kind="num", lo=0, hi=40, options=None, info=False),
]
for _e in _EXTRA_RULES:
    if not any(_s["key"] == _e["key"] for _s in TR_SPEC): TR_SPEC.append(_e)

def _payload(T, L):
    for _s in TR_SPEC:
        if _s["key"] == "leg_length_m": _s["default"] = 250      # short walks: shorter legs
    p = export_payload(T, L); p["tileUrls"] = TILE_URLS; p["tileMaxZoom"] = TILE_MAX_ZOOM
    return p

def _ares_load(slon, slat, elon, elat):
    """Called by the app when START/END are outside the loaded area: fetch real height data around them."""
    global AREA, T, L
    lat_c, lon_c = (slat + elat) / 2, (slon + elon) / 2
    span = max(abs(slat - elat) * M_PER_DEG, abs(slon - elon) * M_PER_DEG * math.cos(math.radians(lat_c)))
    side = min(max(span * 2.2, 8000.0), 800e3)                  # metres: at least 8 km (so nearby picks need no new download), at most 800 km
    dlat = side / 2 / M_PER_DEG; dlon = dlat / max(math.cos(math.radians(lat_c)), 0.05)
    AREA = dict(lat_min=max(lat_c - dlat, -89.5), lat_max=min(lat_c + dlat, 89.5), lon_min=lon_c - dlon, lon_max=lon_c + dlon)
    t0 = time.time(); T = get_terrain(); t1 = time.time(); L = build_layers(T, TR); t2 = time.time()
    pj = _payload(T, L); pj["timing"] = dict(download=round(t1 - t0, 2), layers=round(t2 - t1, 2), pack=round(time.time() - t2, 2))
    print(f"[ares_load] area {side/1000:.0f} km: download {t1-t0:.1f}s, layers {t2-t1:.1f}s, pack {time.time()-t2:.1f}s")
    from IPython.display import JSON
    return JSON(pj)

try:
    from google.colab import output as _co
    _co.register_callback("ares_load", _ares_load)
    print("Colab detected: picking points anywhere on Mars will load real height data for them.")
except Exception:
    print("Not Colab: the app works for the area loaded in cell 4. To use another place, change SITE/CENTER in cell 4 and re-run cells 4-5.")

_page = UI_HTML.replace("__ENGINE__", ENGINE_JS.replace("</script>", "<\\/script>")).replace("__PAYLOAD__", json.dumps(_payload(T, L)))
open("ARES_app.html", "w", encoding="utf-8").write(_page)       # standalone copy (only the first area)
_inline = re.sub(r"(?is)<!DOCTYPE html>|</?html[^>]*>|</?head>|</?body>|<title>.*?</title>|<meta[^>]*>", "", _page)
display(HTML(_inline))
